# 1 — Generation and the cost of the five-color limit

This notebook gives the output for the paper sections **Methodology** (color calibration, color branch, luminance branch, synthesis) and **Results → "Generation and the cost of the five-color limit"**: the table `tab:quant` and the figures `fig_luminance` and `fig_patterns`.

It does these steps again each time you start it:

0. Calibrate the colors of the site photographs with the ColorChecker photograph `IMG_4209`.
1. Make the luminance map of the generation frame `IMG_4171` (figure `fig_luminance`).
2. Make one site-specific pattern for each base pattern (MultiCam, M81, Realtree): luminance map → gray style transfer (2000 steps) → five-color palette → recoloring (figure `fig_patterns`).
3. Measure the local contrast at each stage (the paragraph "The generation step also leaves a mark").
4. The color-limit test: color the same style-transfer output with 5, 8, 16 and 64 colors and with no limit, and color the base pattern the same way without style transfer. Score each version with SINet-V2 and DGNet on 33 scenes × 3 shapes × 20 positions (table `tab:quant`).

**Start the notebook from the project root.** Step 4 takes many hours on the Apple GPU.

**Names.** The code, the folders and the CSV files use `woodland` for the base pattern that the paper calls **M81**.

**Outputs:** `results/01_generation/`: `patterns/<style>/IMG_4171_cal/` (the patterns that notebook 2 tests), `colour_limit/` (results and table), `figures/`, `tables/`. After you start this notebook again, start notebook 2 again too: new patterns make the old notebook 2 results stale.

The Appendix at the end holds the code that made two inputs earlier: the base patterns and the printed sheet `print_v1`.

## Research question and hypothesis half (a)

**Research question.** "To what extent does the imposition of strict color-quantization constraints (k-means) on Neural Style Transfer degrade the adversarial robustness of generated camouflage against established AI detection networks like SINet-V2?"

**Hypothesis** (proposal wording). "While k-means quantization restricts color variance for physical printability, the structural blending from Neural Style Transfer will sufficiently mitigate this penalty, maintaining a significant advantage over generic baselines."

*Measure:* $\Delta_q = S_\alpha(\text{5 colours}) - S_\alpha(\text{no colour limit})$ for each base pattern and detector, paired over 33 backgrounds.
*Rule:* the limit has a real cost in a cell when $\Delta_q > 0$ and the Holm-corrected two-sided Wilcoxon $p < 0.05$.

**Hypothesis half (a).** "The structural blending from Neural Style Transfer will sufficiently mitigate this penalty" (proposal wording). "Sufficiently" is tested as: style transfer removes more than half of the cost.

*Measure:* the same $\Delta_q$ for the base pattern **without** style transfer, and the difference $\Delta_q^{\text{with}} - \Delta_q^{\text{without}}$.
*Rule:* half (a) is supported in a cell when both conditions are true:
1. the difference is negative, with Holm-corrected $p < 0.05$ (style transfer reduces the cost);
2. the compensated share $1 - \Delta_q^{\text{with}} / \Delta_q^{\text{without}}$ is more than 0.5 ("most").

The last cell of this notebook applies these rules.

In [ ]:
import os, sys, json, shutil, csv
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')   # let the detectors use the Apple GPU
from pathlib import Path
import numpy as np, cv2
import matplotlib.pyplot as plt
from PIL import Image
ROOT = Path.cwd()
assert (ROOT/'images').is_dir(), 'start this notebook from the project root'

# ==================== SETTINGS ====================
FRAME     = 'IMG_4171'                          # generation frame
TAG       = '_cal'                              # ColorChecker-calibrated colour
STYLES    = ['multicam', 'woodland', 'realtree']
NAME      = {'multicam': 'MultiCam', 'woodland': 'M81', 'realtree': 'Realtree'}   # 'woodland' is the M81 base
DETECTORS = ('SINet-V2', 'DGNet')
ITERS     = 2000                                # style-transfer steps
DEVICE    = 'mps'                               # 'mps' (Apple GPU) | 'cpu'
FRESH_RUN = True                                # True: erase the earlier outputs of this notebook first
                                                # False: continue an interrupted run

SITE         = ROOT/'images'/'site_photos'
PNG_DIR      = SITE/'calibrated'/'png'
CHART        = 'IMG_4209'                       # the one well-exposed ColorChecker frame
CHART_FRAMES = ['IMG_4174', 'IMG_4193', 'IMG_4194', 'IMG_4198', 'IMG_4204', 'IMG_4209']
OUT          = ROOT/'results'/'01_generation'
PATTERNS     = OUT/'patterns'
FIG          = OUT/'figures'

if FRESH_RUN and OUT.exists():
    shutil.rmtree(OUT)
FIG.mkdir(parents=True, exist_ok=True)
print(f'frame={FRAME}  styles={STYLES}  iters={ITERS}  device={DEVICE}  out={OUT.relative_to(ROOT)}')

## Code

The cells of this section hold all the code that the steps below use. Each code cell is one module, and the text above it names the section of the paper that it implements. Run this section before the steps.

In [ ]:
# A cell with %%module on its first line holds one module of the code. The cell runs as its
# own Python module, so its names do not mix with the names of the notebook (both can have OUT,
# for example). Other cells get a module with a usual import, for example `import adversarial_eval as rv`.
import sys, types, linecache
from IPython.core.magic import register_cell_magic

class CellSource:
    """Gives the cell text to linecache again, so that tracebacks and TensorFlow can read it."""
    def __init__(self, text):
        self.text = text

    def get_source(self, name):
        return self.text

@register_cell_magic
def module(line, cell):
    name = line.strip()
    m = types.ModuleType(name)
    m.__file__ = f'%%module {name}'                    # the file name in tracebacks
    m.__loader__ = CellSource(cell)
    linecache.cache[m.__file__] = (len(cell), None, cell.splitlines(True), m.__file__)
    sys.modules[name] = m
    exec(compile(cell, m.__file__, 'exec', dont_inherit=True), m.__dict__)

### Color calibration — Methodology, "Data Acquisition"

Each RAW file is decoded with rawpy to linear color with sRGB primaries, using the white balance of the chart photograph. The 24 chart patches are found automatically. After the gray patches are scaled to the reference white, a 3×3 matrix is fitted in linear sRGB from the measured patches to the published chart values, and it corrects every photograph. Each photograph is then scaled so that its median brightness matches the same decode with LibRaw's automatic brightness.

The scene photographs are decoded at half size (`half_size=True`): LibRaw joins each 2×2 Bayer block into one pixel and does no demosaicing. Only the chart photograph is decoded at full size with AHD demosaicing.

In [ ]:
%%module calibration
"""Color calibration of the RAW photographs with one ColorChecker chart.

1. The chart photograph is decoded to linear sRGB with the chart's white balance, and the
   24 patches are found automatically.
2. The gray patches are scaled to the reference white, and a 3x3 matrix is fitted in linear
   sRGB from the measured patches to the published chart values.
3. Each other photograph is decoded with the same white balance, corrected with the matrix,
   and scaled so that its median brightness is equal to the same decode with LibRaw's
   automatic brightness. The result is an 8-bit sRGB PNG.
"""
import glob, json, os
from pathlib import Path

import numpy as np, rawpy, cv2

ROOT = Path.cwd()          # project root (the notebook starts here)
N_PATCHES = 24
# colour-science reference data for the chart (the first name that exists is used)
REF_DATASET_CANDIDATES = ["ColorChecker24 - After November 2014",
                          "ColorChecker24 - Before November 2014",
                          "ColorChecker 2005"]


def _fix_wb(wb):
    """Bayer sensors often give 0 for the second green; use the first green there."""
    wb = list(wb)
    if len(wb) >= 4 and wb[3] == 0:
        wb[3] = wb[1]
    return wb


def decode_chart(dng_path):
    """Chart photograph -> linear sRGB in [0, 1] at full size, and its camera white balance."""
    with rawpy.imread(dng_path) as raw:
        cam_wb = _fix_wb(raw.camera_whitebalance)
        rgb16 = raw.postprocess(gamma=(1, 1), no_auto_bright=True, output_bps=16,
                                output_color=rawpy.ColorSpace.sRGB,
                                demosaic_algorithm=rawpy.DemosaicAlgorithm.AHD,
                                highlight_mode=rawpy.HighlightMode.Clip, user_wb=cam_wb)
    return rgb16.astype(np.float32) / 65535.0, cam_wb


def decode(dng, wb, auto_bright, half=True):
    """Scene photograph -> linear sRGB in [0, 1] with the white balance `wb`.
    half=True gives LibRaw's half-size output (each 2x2 Bayer block becomes one pixel)."""
    with rawpy.imread(dng) as raw:
        rgb = raw.postprocess(gamma=(1, 1), no_auto_bright=(not auto_bright), output_bps=16,
                              output_color=rawpy.ColorSpace.sRGB,
                              demosaic_algorithm=rawpy.DemosaicAlgorithm.AHD,
                              highlight_mode=rawpy.HighlightMode.Clip, user_wb=_fix_wb(wb), half_size=half)
    return rgb.astype(np.float32) / 65535.0


def lin_to_lab(lin):
    """Linear sRGB -> CIE Lab (D65)."""
    import colour
    return colour.XYZ_to_Lab(colour.sRGB_to_XYZ(np.clip(np.asarray(lin, float), 0, 1), apply_cctf_decoding=False))


def reference_swatches():
    """The 24 published chart colors as linear sRGB (D65), their Lab values, and the dataset name."""
    import colour
    name = next(n for n in REF_DATASET_CANDIDATES if n in colour.CCS_COLOURCHECKERS)
    ds = colour.CCS_COLOURCHECKERS[name]
    xyY = np.array(list(ds.data.values()))
    lin = colour.XYZ_to_sRGB(colour.xyY_to_XYZ(xyY), illuminant=ds.illuminant, apply_cctf_encoding=False)
    lin = np.clip(lin, 0.0, 1.0).astype(np.float64)
    return lin, lin_to_lab(lin), name


def detect_swatches(lin):
    """Find the chart and return its 24 patch colors as linear sRGB (24 x 3).
    The detector expects a gamma-encoded image, so the image is encoded first and the
    patch colors are decoded back to linear light."""
    from colour_checker_detection import detect_colour_checkers_segmentation
    a = np.clip(lin, 0, 1)
    enc = np.where(a <= 0.0031308, 12.92 * a, 1.055 * np.power(a, 1 / 2.4) - 0.055)
    res = detect_colour_checkers_segmentation(enc.astype(np.float32), additional_data=True)
    if not res:
        raise RuntimeError('ColorChecker not found')
    sw = np.asarray(res[0].swatch_colours, float)
    assert sw.shape == (N_PATCHES, 3), sw.shape
    sw = np.clip(sw, 0, 1)
    return np.where(sw <= 0.04045, sw / 12.92, ((sw + 0.055) / 1.055) ** 2.4)


def luma(x):
    return 0.2126 * x[..., 0] + 0.7152 * x[..., 1] + 0.0722 * x[..., 2]


def save8(path, lin):
    """Linear sRGB -> 8-bit sRGB PNG."""
    a = np.clip(lin, 0, 1)
    enc = np.where(a <= 0.0031308, 12.92 * a, 1.055 * np.power(a, 1 / 2.4) - 0.055)
    cv2.imwrite(str(path), (np.clip(enc * 255 + 0.5, 0, 255).astype(np.uint8))[..., ::-1])


def calibrate(site, chart, out, exclude=()):
    """Calibrate every DNG in `site` with the chart photograph `chart` and write PNGs to `out`.
    `site` and `out` are paths from the project root. The chart and the frames in `exclude`
    get no PNG."""
    site_dir, out_dir = ROOT / site, ROOT / out
    out_dir.mkdir(parents=True, exist_ok=True)
    exclude = set(exclude) | {chart}

    ref_lin, ref_lab, ds = reference_swatches()
    chart_lin, chart_wb = decode_chart(str(site_dir / f"{chart}.DNG"))
    sw = detect_swatches(chart_lin)
    print(f"chart {chart}: detected via auto-detect, dataset {ds}")

    # scale the gray patches (18..23) to the reference white, then fit M so that ref = sw @ M
    s = float(np.median(ref_lin[18:24]) / max(np.median(sw[18:24]), 1e-6))
    M, *_ = np.linalg.lstsq(sw * s, ref_lin, rcond=None)
    apply = lambda img: np.clip(img.reshape(-1, 3) @ M, 0, 1).reshape(img.shape)
    import colour
    de = np.asarray(colour.delta_E(lin_to_lab(np.clip(apply((sw * s)[None])[0], 0, 1)), ref_lab, method="CIE 2000"))
    print(f"gain-normalised linear3x3 CCM: chart dE2000 mean={de.mean():.2f} max={de.max():.2f}")

    frames = sorted(Path(p).stem for p in glob.glob(str(site_dir / "*.DNG")) if os.path.getsize(p) > 0)
    frames = [f for f in frames if f not in exclude]
    for f in frames:
        dng = str(site_dir / f"{f}.DNG")
        corrected = np.clip(apply((decode(dng, chart_wb, auto_bright=False) * s)[None])[0], 0, 1)
        target = float(np.median(luma(decode(dng, chart_wb, auto_bright=True))))
        k = target / max(float(np.median(luma(corrected))), 1e-5)
        save8(out_dir / f"{f}.png", np.clip(corrected * k, 0, 1))
        print(f"  {f}.png")
    json.dump({"site": str(site), "chart": chart, "method": "gainnorm_linear3x3+exposure_norm",
               "chart_dE_mean": float(de.mean()), "frames": frames, "excluded": sorted(exclude)},
              open(out_dir / "calibration_manifest.json", "w"), indent=1)
    print(f"\ncalibrated {len(frames)} frames -> {out_dir}")

### Luminance branch — Methodology, "Luminance Branch"

The generation frame is split into a 3×3 grid. The nine tiles are converted to gray and averaged pixel by pixel, and the average is histogram-equalized. This gray image is the content image for style transfer.

In [ ]:
%%module luminance
"""Luminance branch: the content image for style transfer.

The photograph is cut into a 3x3 grid. The nine tiles are converted to gray and averaged
pixel by pixel, and the average is histogram-equalized.
"""
import numpy as np
from PIL import Image


def equalize_histogram(gray_img):
    """Histogram equalization of an 8-bit gray image."""
    arr = np.asarray(gray_img)
    cdf = np.bincount(arr.ravel(), minlength=256).cumsum()
    cdf_masked = np.ma.masked_equal(cdf, 0)
    lut = (cdf_masked - cdf_masked.min()) * 255 / (cdf_masked.max() - cdf_masked.min())
    lut = np.ma.filled(lut, 0).astype(np.uint8)
    return Image.fromarray(lut[arr], mode="L")


def segment_image(img, rows=3, cols=3):
    """Cut the image into rows x cols tiles that cover it. Returns a list of (tile, box)."""
    xs = np.linspace(0, img.size[0], cols + 1, dtype=int)
    ys = np.linspace(0, img.size[1], rows + 1, dtype=int)
    boxes = [(xs[j], ys[i], xs[j + 1], ys[i + 1]) for i in range(rows) for j in range(cols)]
    return [(img.crop(b), b) for b in boxes]


def luminance_map(img, rows=3, cols=3):
    """Average of the gray tiles, histogram-equalized."""
    tiles = [t[0] for t in segment_image(img, rows, cols)]
    # the tiles can differ by one pixel in size; use the size of the smallest tile
    w, h = min(t.size[0] for t in tiles), min(t.size[1] for t in tiles)
    arrays = [np.asarray(t.convert("L").resize((w, h))) for t in tiles]
    avg = np.mean(arrays, axis=0).astype(np.uint8)
    return equalize_histogram(Image.fromarray(avg, mode="L"))

### Color branch — Methodology, "Color Branch: Five-Color Palette"

Only lit, non-sky pixels are used. Deep shadow is removed (HSV value $V < 60$), and so is sky: pixels that are blue and bright (OpenCV 8-bit Lab, $B < 118$ and $L > 140$), or bright and smooth in the upper half of the frame ($L > 130$ and a local standard deviation below 8 in a 25×25 window). 200,000 of the remaining pixels are clustered with k-means ($k = 5$, ten restarts, fixed seed), with $a^*$ and $b^*$ multiplied by $W = 10$ so that clusters separate by color more than by brightness. The five centers are sorted from dark to light, and each gets the share of pixels in its cluster.

In [ ]:
%%module palette
"""Color branch: the five-color palette of the site.

Only lit, non-sky pixels are used: deep shadow (HSV value V < 60) and sky are removed.
200,000 of the remaining pixels are clustered with k-means in OpenCV's 8-bit Lab
(k = 5, ten restarts, fixed seed). a* and b* are multiplied by W = 10 before clustering,
so that clusters separate by color more than by brightness. The centers are sorted from
dark to light, and each gets the share of the lit pixels in its cluster.
"""
import cv2
import numpy as np
from sklearn.cluster import KMeans

PALETTE_VERSION = 'v6-lit-only'   # the name stored in env_palette.json
CHROMA_WEIGHT = 10.0
K_PALETTE = 5
SHADOW_V_MAX = 60


def detect_sky(rgb):
    """Boolean sky mask: blue and bright (Lab b < 118 and L > 140), or bright and smooth in
    the upper half (L > 130 and a local standard deviation below 8 in a 25 x 25 window).
    The mask is cleaned with a 25 x 25 morphological close and open."""
    bgr = rgb[:, :, ::-1]
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2Lab)
    L_ch, b_ch = lab[..., 0], lab[..., 2]
    H, W, _ = rgb.shape
    blue_sky = (b_ch < 118) & (L_ch > 140)
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY).astype(np.float32)
    blur = cv2.blur(gray, (25, 25))
    local_std = np.sqrt(cv2.blur((gray - blur) ** 2, (25, 25)))
    yy = np.arange(H).reshape(-1, 1).repeat(W, axis=1)
    overcast = (local_std < 8) & (L_ch > 130) & (yy < H * 0.5)
    sky = (blue_sky | overcast).astype(np.uint8) * 255
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (25, 25))
    sky = cv2.morphologyEx(sky, cv2.MORPH_CLOSE, kernel)
    sky = cv2.morphologyEx(sky, cv2.MORPH_OPEN, kernel)
    return sky > 0


def lit_only_palette(env_path, k=K_PALETTE, chroma_weight=CHROMA_WEIGHT, shadow_v_max=SHADOW_V_MAX,
                     n_subsample=200_000, random_state=42):
    """k Lab centers (sorted by L) and their pixel shares, from the lit, non-sky pixels.
    Also returns the percent of sky, shadow and lit pixels."""
    bgr = cv2.imread(str(env_path))
    if bgr is None:
        raise FileNotFoundError(env_path)
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2Lab).astype(np.float32)
    sky_mask = detect_sky(rgb)
    shadow_mask = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)[..., 2] < shadow_v_max
    lit_mask = ~(sky_mask | shadow_mask)
    sky_pct, shadow_pct, lit_pct = (m.sum() / m.size * 100.0 for m in (sky_mask, shadow_mask, lit_mask))

    lit_pix = lab.reshape(-1, 3)[lit_mask.ravel()]
    if lit_pix.shape[0] > n_subsample:
        idx = np.random.RandomState(random_state).choice(lit_pix.shape[0], n_subsample, replace=False)
        sample = lit_pix[idx]
    else:
        sample = lit_pix

    def weight_ab(arr, w):
        out = arr.copy().astype(np.float32)
        out[:, 1:] = (out[:, 1:] - 128) * w + 128
        return out

    km = KMeans(n_clusters=k, n_init=10, random_state=random_state).fit(weight_ab(sample, chroma_weight))
    cents = km.cluster_centers_.copy()
    cents[:, 1:] = (cents[:, 1:] - 128) / chroma_weight + 128
    counts = np.bincount(km.predict(weight_ab(lit_pix, chroma_weight)), minlength=k)
    fractions = counts / counts.sum()
    order = np.argsort(cents[:, 0])
    return cents[order], fractions[order], sky_pct, shadow_pct, lit_pct

### Recoloring — Methodology, "Synthesis: Style Transfer and Recoloring"

The pixels of the gray style-transfer output are sorted by brightness. The darkest share $f_1$ of the pixels gets the darkest palette color, the next share $f_2$ the next color, and so on, where $f_i$ is the share of cluster $i$. The pattern therefore has exactly five colors.

In [ ]:
%%module recolor
"""Recoloring of the gray style-transfer output with the site palette.

The pixels are sorted by brightness. The darkest share f1 of the pixels gets the darkest
palette color, the next share f2 the next color, and so on, where fi is the share of
cluster i. The result has exactly as many colors as the palette.
"""
import cv2
import numpy as np


def recolor_frequency_weighted(gray_rgb, lab_centroids_sorted_by_L, fractions_sorted_by_L):
    H, W, _ = gray_rgb.shape
    L = gray_rgb[..., 0].astype(np.float32).ravel()
    N = L.size
    cum = (np.cumsum(fractions_sorted_by_L) * N).astype(np.int64)   # last pixel index of each color
    cum[-1] = N
    order = np.argsort(L, kind='stable')
    cluster_id = np.empty(N, dtype=np.int32)
    start = 0
    for i, end in enumerate(cum):
        cluster_id[order[start:end]] = i
        start = end
    lab_img = np.clip(lab_centroids_sorted_by_L[cluster_id].reshape(H, W, 3), 0, 255).astype(np.uint8)
    return cv2.cvtColor(cv2.cvtColor(lab_img, cv2.COLOR_Lab2BGR), cv2.COLOR_BGR2RGB)

### Style transfer and the generation of one pattern — Methodology, "Synthesis: Style Transfer and Recoloring"

Style transfer (Gatys et al.) uses the luminance map as content and a gray copy of the base pattern as style. Style comes from the first layer of each of the five VGG-19 blocks and content from `block5_conv2`, with a style weight of $10^{-6}$ and a content weight of $2.5\times10^{-8}$. The optimization starts from the content image and runs plain gradient descent on the image for 2000 steps; the learning rate starts at 100 and falls by 4 % every 100 steps. Seeds are fixed, and the output is 400×300 pixels. `generate()` then makes the five-color palette and recolors the gray output.

In [ ]:
%%module generate_pattern
"""Synthesis: style transfer and recoloring. Makes one site-specific pattern.

    luminance map of the photograph (content) + gray base pattern (style)
    -> style transfer (Gatys et al., VGG-19, 2000 steps) -> gray output
    -> five-color palette of the photograph -> recoloring by brightness rank

Style comes from the first layer of each of the five VGG-19 blocks and content from
block5_conv2, with a style weight of 1e-6 and a content weight of 2.5e-8. The optimization
starts from the content image and runs plain gradient descent on the image. The learning
rate starts at 100 and falls by 4 % every 100 steps. The output is 400 x 300 pixels.

Output files in out_dir: final_camo.png (the pattern), palette_swatch.png (its colors),
nst_gray.png, luminance_map.png and env_palette.json.
"""
import json
import time
from pathlib import Path

import cv2
import numpy as np
from PIL import Image

from luminance import luminance_map
from palette import lit_only_palette, PALETTE_VERSION
from recolor import recolor_frequency_weighted

ROOT = Path.cwd()          # project root (the notebook starts here)
BASE_CAMO_DIR = ROOT / 'images' / 'base_patterns'

NST_HEIGHT     = 400
STYLE_LAYERS   = ['block1_conv1', 'block2_conv1', 'block3_conv1', 'block4_conv1', 'block5_conv1']
CONTENT_LAYER  = 'block5_conv2'
CONTENT_WEIGHT = 2.5e-8
STYLE_WEIGHT   = 1e-6


def build_feature_extractor():
    """VGG-19 (ImageNet weights) that returns the output of every layer."""
    import tensorflow as tf
    from keras import Model
    from keras.applications import vgg19
    vgg = vgg19.VGG19(weights='imagenet', include_top=False)
    outputs = {layer.name: layer.output for layer in vgg.layers}
    return tf, vgg19, Model(inputs=vgg.inputs, outputs=outputs)


def grayscale_camo(camo_path):
    """The base pattern in gray, as a 3-channel image (its color is discarded)."""
    gray = cv2.cvtColor(cv2.imread(str(camo_path)), cv2.COLOR_BGR2GRAY)
    return cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)


def force_grayscale(rgb):
    bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
    return cv2.cvtColor(cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY), cv2.COLOR_GRAY2RGB)


def run_nst(tf, vgg19, feature_extractor, content_pil, style_arr, iterations):
    """Style transfer of style_arr onto content_pil. Returns an RGB uint8 image."""
    from keras.optimizers import SGD

    def gram(x):
        x = tf.transpose(x, (2, 0, 1))
        f = tf.reshape(x, (tf.shape(x)[0], -1))
        return tf.matmul(f, tf.transpose(f))

    def style_loss_fn(sf, cf, nrows, ncols):
        S, C = gram(sf), gram(cf)
        size = nrows * ncols
        return tf.reduce_sum(tf.square(S - C)) / (4.0 * (3 ** 2) * (size ** 2))

    def total_loss(combination, base, style, nrows, ncols):
        feats = feature_extractor(tf.concat([base, style, combination], axis=0))
        cf = feats[CONTENT_LAYER]
        loss = tf.constant(0.0)
        loss = loss + CONTENT_WEIGHT * tf.reduce_sum(tf.square(cf[2] - cf[0]))
        for name in STYLE_LAYERS:
            sf = feats[name]
            loss = loss + (STYLE_WEIGHT / len(STYLE_LAYERS)) * style_loss_fn(sf[1], sf[2], nrows, ncols)
        return loss

    @tf.function
    def loss_and_grads(combination, base, style, nrows, ncols):
        with tf.GradientTape() as tape:
            loss = total_loss(combination, base, style, nrows, ncols)
        return loss, tape.gradient(loss, combination)

    def preprocess(pil_img, h, w):
        arr = np.array(pil_img.convert('RGB').resize((w, h)), dtype=np.float32)[None]
        return tf.convert_to_tensor(vgg19.preprocess_input(arr))

    def deprocess(x, h, w):
        x = x.reshape((h, w, 3)).copy()
        x[:, :, 0] += 103.939; x[:, :, 1] += 116.779; x[:, :, 2] += 123.68
        return np.clip(x[:, :, ::-1], 0, 255).astype(np.uint8)

    content_w, content_h = content_pil.size
    nrows = NST_HEIGHT
    ncols = int(content_w * nrows / content_h)
    base_tensor = preprocess(content_pil, nrows, ncols)
    style_tensor = preprocess(Image.fromarray(style_arr), nrows, ncols)
    combination = tf.Variable(base_tensor)
    optimizer = SGD(tf.keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=100.0, decay_steps=100, decay_rate=0.96))
    t0 = time.time()
    for i in range(1, iterations + 1):
        loss, grads = loss_and_grads(combination, base_tensor, style_tensor, nrows, ncols)
        optimizer.apply_gradients([(grads, combination)])
        if i % 100 == 0:
            print(f'  iter {i:5d}/{iterations}: loss={loss.numpy():.2f} ({(time.time()-t0)/60:.1f} min)', flush=True)
    return deprocess(combination.numpy(), nrows, ncols)


def generate(photo, base, iterations, out_dir):
    """Make the site-specific pattern for base pattern `base` from the calibrated PNG `photo`."""
    photo, out_dir = Path(photo), Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    base_path = BASE_CAMO_DIR / f'{base}.jpg'

    lum_map = luminance_map(Image.open(photo).convert('RGB'), rows=3, cols=3)
    lum_map.save(out_dir / 'luminance_map.png')

    print(f'style transfer: {photo.name} (content) + {base_path.name} (style), {iterations} steps')
    tf, vgg19, feature_extractor = build_feature_extractor()
    np.random.seed(42); tf.random.set_seed(42)
    nst_color = run_nst(tf, vgg19, feature_extractor, lum_map, grayscale_camo(base_path), iterations)
    nst_gray = force_grayscale(nst_color)
    Image.fromarray(nst_gray).save(out_dir / 'nst_gray.png')

    cents, fracs, sky_pct, shadow_pct, lit_pct = lit_only_palette(photo)
    print(f'palette: sky={sky_pct:.1f}%  shadow(V<60)={shadow_pct:.1f}%  lit={lit_pct:.1f}%')
    final = recolor_frequency_weighted(nst_gray, cents, fracs)
    Image.fromarray(final).save(out_dir / 'final_camo.png')

    hexes = []
    sw_w = 80
    swatch = np.zeros((80, sw_w * len(cents), 3), dtype=np.uint8)
    for i, c in enumerate(cents):
        rgb = cv2.cvtColor(np.uint8([[np.clip(c, 0, 255)]]), cv2.COLOR_Lab2RGB)[0][0]
        swatch[:, i * sw_w:(i + 1) * sw_w] = rgb
        hexes.append(f'#{rgb[0]:02x}{rgb[1]:02x}{rgb[2]:02x}')
    Image.fromarray(swatch).save(out_dir / 'palette_swatch.png')

    with open(out_dir / 'env_palette.json', 'w') as f:
        json.dump({'env': photo.name, 'base_pattern': base_path.name, 'iterations': iterations,
                   'palette_version': PALETTE_VERSION,
                   'k': len(cents), 'sky_masked_percent': sky_pct,
                   'shadow_masked_percent': shadow_pct, 'lit_remaining_percent': lit_pct,
                   'palette_hex_sorted_by_L': hexes,
                   'lab_centroids_sorted_by_L': cents.tolist(),
                   'pixel_fractions_sorted_by_L': fracs.tolist()}, f, indent=2)
    print(f'-> {out_dir / "final_camo.png"}  palette {"  ".join(hexes)}')
    return out_dir / 'final_camo.png'

### Target shapes — Evaluation, "Digital Evaluation"

The three silhouettes that are pasted into the test scenes. The human silhouette is 0.35 of the frame height; the animal and vehicle bodies are 0.30 and 0.35 of the frame width.

In [ ]:
%%module silhouettes
"""The three target shapes that are pasted into the test scenes.

The human silhouette is 0.35 of the frame height; the animal and vehicle bodies are 0.30
and 0.35 of the frame width. Each function returns a uint8 mask (255 = target).
"""
import numpy as np
import cv2


class SilhouetteLibrary:
    CLASSES = ['quadruped', 'human', 'vehicle']

    @classmethod
    def quadruped(cls, h, w, scale=0.30):
        m = np.zeros((h, w), dtype=np.uint8)
        cy = int(h * 0.62)
        body_w = int(w * scale)
        body_h = int(body_w * 0.45)
        cx = w // 2
        cv2.ellipse(m, (cx, cy), (body_w // 2, body_h // 2), 0, 0, 360, 255, -1)
        head_r = int(body_h * 0.55)
        head_cx = cx + body_w // 2 - head_r // 2
        head_cy = cy - body_h // 2 - head_r // 3
        cv2.circle(m, (head_cx, head_cy), head_r, 255, -1)
        leg_w = max(2, int(body_w * 0.06))
        leg_h = int(body_h * 0.95)
        for lx in (cx - body_w // 3, cx - body_w // 6, cx + body_w // 6, cx + body_w // 3):
            cv2.ellipse(m, (lx, cy + body_h // 3), (leg_w, leg_h // 2), 0, 0, 360, 255, -1)
        tail_pts = np.array([
            [cx - body_w // 2, cy - body_h // 4],
            [cx - body_w // 2 - int(body_w * 0.15), cy - body_h // 2],
            [cx - body_w // 2 - int(body_w * 0.12), cy - body_h // 4],
        ], dtype=np.int32)
        cv2.fillPoly(m, [tail_pts], 255)
        return m

    @classmethod
    def human(cls, h, w, scale=0.35):
        m = np.zeros((h, w), dtype=np.uint8)
        fig_h = int(h * scale)
        cx = w // 2
        cy = h // 2 + fig_h // 6
        head_r = max(4, int(fig_h * 0.08))
        head_cy = cy - fig_h // 2 + head_r
        cv2.circle(m, (cx, head_cy), head_r, 255, -1)
        torso_w = int(fig_h * 0.22)
        torso_h = int(fig_h * 0.35)
        torso_y0 = head_cy + head_r
        cv2.rectangle(m, (cx - torso_w // 2, torso_y0), (cx + torso_w // 2, torso_y0 + torso_h), 255, -1)
        arm_w = max(3, int(fig_h * 0.05))
        arm_h = int(fig_h * 0.32)
        cv2.rectangle(m, (cx - torso_w // 2 - arm_w, torso_y0), (cx - torso_w // 2, torso_y0 + arm_h), 255, -1)
        cv2.rectangle(m, (cx + torso_w // 2, torso_y0), (cx + torso_w // 2 + arm_w, torso_y0 + arm_h), 255, -1)
        leg_w = max(3, int(fig_h * 0.06))
        leg_h = int(fig_h * 0.40)
        leg_y0 = torso_y0 + torso_h
        cv2.rectangle(m, (cx - torso_w // 4 - leg_w, leg_y0), (cx - torso_w // 4, leg_y0 + leg_h), 255, -1)
        cv2.rectangle(m, (cx + torso_w // 4, leg_y0), (cx + torso_w // 4 + leg_w, leg_y0 + leg_h), 255, -1)
        return m

    @classmethod
    def vehicle(cls, h, w, scale=0.35):
        m = np.zeros((h, w), dtype=np.uint8)
        body_w = int(w * scale)
        body_h = int(body_w * 0.32)
        cx = w // 2
        cy = int(h * 0.60)
        body_y0 = cy - body_h // 2
        cv2.rectangle(m, (cx - body_w // 2, body_y0), (cx + body_w // 2, body_y0 + body_h), 255, -1)
        cab_pts = np.array([
            [cx - body_w // 4, body_y0],
            [cx - body_w // 6, body_y0 - int(body_h * 0.7)],
            [cx + body_w // 5, body_y0 - int(body_h * 0.7)],
            [cx + body_w // 3, body_y0],
        ], dtype=np.int32)
        cv2.fillPoly(m, [cab_pts], 255)
        wheel_r = int(body_h * 0.55)
        wheel_y = body_y0 + body_h
        cv2.circle(m, (cx - body_w // 3, wheel_y), wheel_r, 255, -1)
        cv2.circle(m, (cx + body_w // 3, wheel_y), wheel_r, 255, -1)
        return m

    @classmethod
    def get(cls, name, h, w):
        return getattr(cls, name)(h, w)

### Scores — Evaluation, "Digital Evaluation"

The paper uses the structure measure $S_\alpha$ (Fan et al. 2017), which compares the whole prediction map with the target mask. A lower $S_\alpha$ means that the detector found the target less well. The other scores ($E_\phi$, $F^w_\beta$, MAE) are stored in the result files but are not used in the paper.

In [ ]:
%%module cod_metrics
"""Detector scores. S_alpha (structure measure, Fan et al. 2017) is the score the paper uses.

    S_alpha - structure measure              (Fan et al., ICCV 2017)
    E_phi   - enhanced-alignment measure     (Fan et al., IJCAI 2018)
    Fw_beta - weighted F-measure             (Margolin et al., CVPR 2014)
    MAE     - mean absolute error

The functions are line-for-line ports of the MATLAB files in models/SINet-V2/eval/
(S_object.m, S_region.m, Enhancedmeasure.m, original_WFb.m). MATLAB std and var use N-1,
so every variance here uses ddof=1.
"""
import numpy as np
import cv2
from scipy.ndimage import distance_transform_edt as _bwdist

_EPS = np.finfo(np.float64).eps


def _matlab_round(v: float) -> int:
    """MATLAB round(): half away from zero. Python's round() is half to even."""
    return int(np.floor(v + 0.5)) if v >= 0 else int(np.ceil(v - 0.5))


class CODMetrics:
    # ------------------------------------------------------------------ MAE
    @staticmethod
    def mae(pred: np.ndarray, gt: np.ndarray) -> float:
        return float(np.mean(np.abs(pred.astype(np.float64) - gt.astype(np.float64))))

    # -------------------------------------------------------------- S-measure
    @staticmethod
    def _object(pred_region: np.ndarray) -> float:
        """S_object.m: score = 2x / (x^2 + 1 + sigma_x + eps)."""
        if pred_region.size == 0:
            return 0.0
        x = float(pred_region.mean())
        sigma_x = float(pred_region.std(ddof=1)) if pred_region.size > 1 else 0.0
        return float(2.0 * x / (x * x + 1.0 + sigma_x + _EPS))

    @classmethod
    def s_object(cls, pred: np.ndarray, gt: np.ndarray) -> float:
        fg = np.where(gt, pred, 0.0)
        bg = np.where(gt, 0.0, 1.0 - pred)
        u = float(gt.mean())
        return u * cls._object(fg[gt]) + (1.0 - u) * cls._object(bg[~gt])

    @staticmethod
    def _ssim(pred: np.ndarray, gt: np.ndarray) -> float:
        """S_region.m's ssim helper, sample variance (N-1)."""
        n = pred.size
        if n == 0:
            return 0.0
        if n == 1:
            # S_region.m: for N==1 all three variances are exactly 0, so alpha==0 and
            # beta==0 and the function takes its Q=1.0 branch regardless of the values.
            return 1.0
        x, y = float(pred.mean()), float(gt.mean())
        sx2 = float(((pred - x) ** 2).sum() / (n - 1 + _EPS))
        sy2 = float(((gt - y) ** 2).sum() / (n - 1 + _EPS))
        sxy = float(((pred - x) * (gt - y)).sum() / (n - 1 + _EPS))
        a = 4.0 * x * y * sxy
        b = (x * x + y * y) * (sx2 + sy2)
        if a != 0.0:
            return float(a / (b + _EPS))
        return 1.0 if (a == 0.0 and b == 0.0) else 0.0

    @classmethod
    def s_region(cls, pred: np.ndarray, gt: np.ndarray) -> float:
        """S_region.m: 4 quadrants about the GT centroid, area-weighted SSIM."""
        h, w = gt.shape
        total = float(gt.sum())
        if total == 0:
            return 0.0
        gtf = gt.astype(np.float64)
        # centroid, MATLAB 1-based
        # MATLAB round() is half-AWAY-from-zero; Python's round() is half-to-EVEN.
        # They disagree on every exact .5 centroid with an even integer part, which
        # shifts the quadrant split by one pixel (audited 2026-09-20: worst case 0.50
        # on 2x2 objects, but 0/180 of this pipeline's silhouette masks ever tie).
        X = _matlab_round(float((np.arange(1, w + 1) * gtf.sum(axis=0)).sum() / total))
        Y = _matlab_round(float((np.arange(1, h + 1) * gtf.sum(axis=1)).sum() / total))
        # DELIBERATE DEVIATION: S_region.m does not clamp, so a centroid on the last
        # row/column gives an empty quadrant, mean2([]) = NaN, and the whole measure
        # returns NaN. We clamp to keep the pipeline finite. Verified never to bind on
        # this corpus (the centroid is interior for every mask used here).
        X = min(max(X, 1), w - 1)
        Y = min(max(Y, 1), h - 1)
        quads = [(slice(0, Y), slice(0, X)), (slice(0, Y), slice(X, w)),
                 (slice(Y, h), slice(0, X)), (slice(Y, h), slice(X, w))]
        area = float(w * h)
        w1 = (X * Y) / area
        w2 = ((w - X) * Y) / area
        w3 = (X * (h - Y)) / area
        w4 = 1.0 - w1 - w2 - w3
        return float(sum(wi * cls._ssim(pred[q].astype(np.float64), gtf[q])
                         for wi, q in zip((w1, w2, w3, w4), quads)))

    @classmethod
    def s_measure(cls, pred: np.ndarray, gt: np.ndarray, alpha: float = 0.5) -> float:
        """StructureMeasure.m"""
        pred = pred.astype(np.float64)
        gtb = gt > 0.5
        y = float(gtb.mean())
        if y == 0.0:
            return float(1.0 - pred.mean())
        if y == 1.0:
            return float(pred.mean())
        q = alpha * cls.s_object(pred, gtb) + (1.0 - alpha) * cls.s_region(pred, gtb)
        return float(max(q, 0.0))

    # -------------------------------------------------------------- E-measure
    @staticmethod
    def _enhanced(binary_pred: np.ndarray, gt: np.ndarray) -> float:
        """Enhancedmeasure.m. Its first line is `FM = logical(FM)`, so the reference
        takes a BINARY prediction. main.m:97-102 binarizes at min(2*mean(cam), 1)
        before calling it. Feeding a continuous sigmoid map here (as the pre-2026-09-20
        code did) is not what the reference computes."""
        pred = binary_pred.astype(np.float64)
        gtf = (gt > 0.5).astype(np.float64)
        h, w = gtf.shape
        s = gtf.sum()
        if s == 0.0:
            enhanced = 1.0 - pred
        elif s == gtf.size:
            enhanced = pred.copy()
        else:
            d_pred = pred - pred.mean()
            d_gt = gtf - gtf.mean()
            align = 2.0 * (d_gt * d_pred) / (d_gt * d_gt + d_pred * d_pred + _EPS)
            enhanced = ((align + 1.0) ** 2) / 4.0
        return float(enhanced.sum() / (w * h - 1 + _EPS))

    @classmethod
    def e_measure(cls, pred: np.ndarray, gt: np.ndarray) -> float:
        """Adaptive E-measure, main.m:97-102: binarize at min(2*mean(pred), 1)."""
        p = pred.astype(np.float64)
        thr = min(2.0 * float(p.mean()), 1.0)
        return cls._enhanced(p > thr, gt)

    # ------------------------------------------------------------- weighted F
    # The distance transform and the importance map depend ONLY on the GT mask, not on
    # the prediction, but the reference recomputes them per call. In this pipeline the
    # same mask is scored 8 times (2 conditions x 2 detectors x 2 normalisations), so a
    # tiny cache keyed on the mask cuts the dominant cost without changing any number.
    _wfb_cache: dict = {}

    @classmethod
    def _wfb_geometry(cls, gtb: np.ndarray):
        key = (gtb.shape, gtb.tobytes())
        hit = cls._wfb_cache.get(key)
        if hit is None:
            dst, idx = _bwdist(~gtb, return_indices=True)
            b = np.where(gtb, 1.0, 2.0 - np.exp(np.log(1 - 0.5) / 5.0 * dst))
            # int32 indices (exact for any real image) halve the entry to ~36 MB,
            # and the bound is 2 because the access pattern is 8 consecutive calls on one
            # mask (2 conditions x 2 detectors x 2 normalisations) before the mask changes.
            hit = (idx.astype(np.int32), b)   # int32 indices are exact; b stays float64
            if len(cls._wfb_cache) >= 2:
                cls._wfb_cache.clear()
            cls._wfb_cache[key] = hit
        return hit

    @classmethod
    def weighted_f_measure(cls, pred: np.ndarray, gt: np.ndarray, beta: float = 1.0) -> float:
        """original_WFb.m (Margolin et al. 2014)."""
        pred = pred.astype(np.float64)
        gtb = gt > 0.5
        if not gtb.any():
            return 0.0
        e = np.abs(pred - gtb.astype(np.float64))
        idx, b = cls._wfb_geometry(gtb)
        # Et: pixel dependency - background error inherits its nearest GT pixel's error
        et = e.copy()
        et[~gtb] = e[idx[0][~gtb], idx[1][~gtb]]
        k = cv2.getGaussianKernel(7, 5.0)
        ea = cv2.filter2D(et, -1, k @ k.T, borderType=cv2.BORDER_CONSTANT)
        min_e_ea = e.copy()
        sel = gtb & (ea < e)
        min_e_ea[sel] = ea[sel]
        ew = min_e_ea * b          # b: pixel importance, from the cached geometry
        tpw = float(gtb.sum()) - float(ew[gtb].sum())
        fpw = float(ew[~gtb].sum())
        r = 1.0 - float(ew[gtb].mean())
        p = tpw / (tpw + fpw + _EPS)
        b2 = beta ** 2
        return float((1.0 + b2) * r * p / (b2 * p + r + _EPS))

    # ------------------------------------------------------------------ all
    @classmethod
    def evaluate_all(cls, pred: np.ndarray, gt: np.ndarray) -> dict:
        p = pred.astype(np.float64)
        g = gt.astype(np.float64)
        if p.max() > 1.01:
            p /= 255.0
        if g.max() > 1.01:
            g /= 255.0
        # main.m:89 min-max stretches every prediction to [0,1] before scoring.
        # That convention makes a uniformly-near-zero map (a total miss) look like a
        # confident detection, which is exactly the wrong behaviour for camouflage
        # evaluation, so the UNNORMALISED values are primary here and the benchmark
        # convention is reported alongside for comparability.
        rng = float(p.max() - p.min())
        pn = (p - p.min()) / rng if rng > 1e-12 else np.zeros_like(p)
        return {
            'MAE':          cls.mae(p, g),
            'S_alpha':      cls.s_measure(p, g),
            'E_phi':        cls.e_measure(p, g),
            'Fw_beta':      cls.weighted_f_measure(p, g),
            'S_alpha_norm': cls.s_measure(pn, g),
            'E_phi_norm':   cls.e_measure(pn, g),
            'Fw_beta_norm': cls.weighted_f_measure(pn, g),
        }

### The two detectors — Evaluation, "Digital Evaluation"

SINet-V2 and DGNet act as fixed judges. Both are used exactly as published, with no retraining; their code and weights are in `models/`.

In [ ]:
%%module detectors
"""The two camouflaged object detectors, used as published with no retraining.

    SINet-V2  Fan et al. 2022, IEEE TPAMI   https://github.com/GewelsJI/SINet-V2
    DGNet     Ji et al. 2023, MIR           https://github.com/GewelsJI/DGNet (EfficientNet-B4)

Each detector takes an RGB uint8 image and returns a float32 map in [0, 1] at the image
size: the sigmoid of its output after a resize of the input to 352 x 352.

torch loads only when build_detectors() runs. When torch and torchvision load before the
k-means palette step in the same process, the palette changes (3 pixels of the M81
pattern in a test on 2026-09-28), so the import must stay inside the function.
"""
import sys, warnings
from pathlib import Path

import numpy as np

ROOT = Path.cwd()          # project root (the notebook starts here)
SINET_WEIGHTS = ROOT / 'models' / 'SINet-V2' / 'snapshot' / 'SINet_V2' / 'Net_epoch_best.pth'
DGNET_WEIGHTS = ROOT / 'models' / 'DGNet' / 'lib_pytorch' / 'snapshot' / 'DGNet.pth'


def _import_repo(repo_dir):
    """Both repositories have a top-level package `lib`. Remove the one that is loaded and
    put this repository first on the path."""
    for mod in [m for m in list(sys.modules) if m == 'lib' or m.startswith('lib.')]:
        del sys.modules[mod]
    sys.path.insert(0, str(repo_dir))


class Detector:
    def __init__(self, model, device, pick):
        import torch
        import torchvision.transforms as T
        self.torch, self.model, self.device, self.pick = torch, model.to(device).eval(), device, pick
        self.transform = T.Compose([T.ToPILImage(), T.Resize((352, 352)), T.ToTensor(),
                                    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])

    def predict(self, image):
        torch = self.torch
        h, w = image.shape[:2]
        x = self.transform(image).unsqueeze(0).to(self.device)
        with torch.no_grad():
            pred = torch.sigmoid(self.pick(self.model(x)))
            pred = torch.nn.functional.interpolate(pred, size=(h, w), mode='bilinear', align_corners=True)
        return pred.squeeze().cpu().numpy().astype(np.float32)


def build_detectors(device):
    """Returns (SINet-V2, DGNet) on `device` ('mps' or 'cpu')."""
    import torch
    device = torch.device(device)

    _import_repo(SINET_WEIGHTS.resolve().parents[2])
    from lib.Network_Res2Net_GRA_NCD import Network
    sinet = Network(channel=32, imagenet_pretrained=False)
    sinet.load_state_dict(torch.load(str(SINET_WEIGHTS), map_location=device, weights_only=False))
    print(f'[SINetV2] Official model loaded — device={device}')

    _import_repo(DGNET_WEIGHTS.resolve().parents[1])
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        from lib.DGNet import DGNet
    dgnet = DGNet(channel=64, arc='EfficientNet-B4', M=[8, 8, 8], N=[4, 8, 16])
    # strict=False: the two extra keys are the classification head of the backbone, which DGNet does not use
    missing, _ = dgnet.load_state_dict(torch.load(str(DGNET_WEIGHTS), map_location=device, weights_only=False), strict=False)
    assert not missing, f'DGNet: {len(missing)} missing keys'
    print(f'[DGNet] Model loaded (EfficientNet-B4, channel=64) — device={device}')

    # SINet-V2 returns several maps, the last is the prediction; DGNet returns (prediction, gradient map)
    return (Detector(sinet, device, lambda out: out[-1] if isinstance(out, (list, tuple)) else out),
            Detector(dgnet, device, lambda out: out[0] if isinstance(out, (tuple, list)) else out))

### Compositing and the detector run — Evaluation, "Compositing Protocol"

Each test pastes one silhouette into a test scene, fills it with a pattern, and asks both detectors to find it. The pattern is tiled so that one repeat is one third of the silhouette height. Each silhouette is moved by up to 10 % of the frame in 20 positions, and both conditions use the same positions. The base pattern is first cropped to the shape of the site-specific pattern and reduced to the same 400×300 size, so that the two differ in design and not in image size.

In [ ]:
%%module adversarial_eval
"""Digital evaluation: paste a silhouette into a test scene, fill it with a pattern, and ask
both detectors to find it.

Compositing protocol: the pattern is tiled so that one repeat is one third of the
silhouette height, and the silhouette edge is feathered (Gaussian, sigma 3 px). Each
silhouette is moved by up to 10 % of the frame in 20 positions. The positions come from a
seed of (base, scene, shape), so the site-specific pattern and its base pattern, and every
arm of the color-limit test, use the same 20 positions. The base pattern is first cropped to
the shape of the site-specific pattern and reduced to its 400 x 300 size
(equal_resolution_base), so the two differ in design and not in image size.

run_eval() scores the base pattern ('generic') and the site-specific pattern ('site') on
every test scene and writes one row per detector run to results.csv. An interrupted run
continues where it stopped.
"""
import csv
import hashlib
import time
from pathlib import Path

import cv2
import numpy as np

from silhouettes import SilhouetteLibrary
from cod_metrics import CODMetrics
from detectors import build_detectors

ROOT = Path.cwd()          # project root (the notebook starts here)
CAMO_ROOT = ROOT / 'results' / '01_generation' / 'patterns'
EVAL_ROOT = ROOT / 'results' / '02_adversarial_test'
BASE_DIR  = ROOT / 'images' / 'base_patterns'

PLACEMENT_FRAC = 0.10     # largest move of the silhouette, as a fraction of the frame
REPEAT_FRAC    = 1/3      # height of one pattern repeat, as a fraction of the silhouette height
METRICS        = ['S_alpha', 'E_phi', 'Fw_beta', 'MAE', 'S_alpha_norm', 'E_phi_norm', 'Fw_beta_norm']
# detector output inside and outside the target, and its highest value
EXTRA_COLS     = ['pred_in_gt', 'pred_out_gt', 'pred_max']
# ColorChecker photographs: calibration inputs, never test scenes
CHART_FRAMES = {'IMG_4174', 'IMG_4193', 'IMG_4194', 'IMG_4198', 'IMG_4204', 'IMG_4209'}


def seed_of(*parts):
    return int(hashlib.md5('|'.join(parts).encode()).hexdigest(), 16) % (2 ** 32)


def load_rgb(p):
    """Image file -> RGB uint8 (16-bit, gray and 4-channel files are converted)."""
    bgr = cv2.imread(str(p), cv2.IMREAD_UNCHANGED)
    if bgr is None:
        raise FileNotFoundError(p)
    if bgr.dtype == np.uint16:
        bgr = (bgr / 257.0).round().astype(np.uint8)
    if bgr.ndim == 2:
        bgr = cv2.cvtColor(bgr, cv2.COLOR_GRAY2BGR)
    if bgr.shape[2] == 4:
        bgr = bgr[..., :3]
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


def composite(camo, bg, mask):
    """Tile `camo` into the silhouette `mask` on the scene `bg`. Returns the image and the
    target mask (float, 1 = target)."""
    h, w = bg.shape[:2]
    sil_h = max(int((mask > 127).any(axis=1).sum()), 1)   # rows that hold the silhouette
    th = max(int(sil_h * REPEAT_FRAC), 8)
    tw = max(int(th * camo.shape[1] / camo.shape[0]), 8)  # keep the aspect ratio of the pattern
    camo = np.tile(cv2.resize(camo, (tw, th)), (h // th + 1, w // tw + 1, 1))[:h, :w]
    a = cv2.GaussianBlur(mask.astype(np.float32), (0, 0), 3.0)
    a = a / max(a.max(), 1e-6)
    out = (a[..., None] * camo + (1 - a[..., None]) * bg).astype(np.uint8)
    return out, (mask > 127).astype(np.float32)


def equal_resolution_base(style, frame='IMG_4171', tag='_cal'):
    """The base pattern cropped to the shape of the style-transfer output and reduced to its
    size (400 x 300, cv2.INTER_AREA), so the two patterns differ in design and not in image size."""
    h, w = cv2.imread(str(CAMO_ROOT / style / (frame + tag) / 'final_camo.png')).shape[:2]
    base = cv2.cvtColor(cv2.imread(str(BASE_DIR / f'{style}.jpg')), cv2.COLOR_BGR2RGB)
    bh, bw = base.shape[:2]
    cw = min(bw, round(bh * w / h)); ch = min(bh, round(cw * h / w))
    y0, x0 = (bh - ch) // 2, (bw - cw) // 2
    return cv2.resize(base[y0:y0 + ch, x0:x0 + cw], (w, h), interpolation=cv2.INTER_AREA)


def placements(style, stem, shape, n_trials, h, w):
    """The n_trials (dx, dy) moves of one silhouette in one scene."""
    rng = np.random.RandomState(seed_of(style, stem, shape))
    max_dx, max_dy = int(w * PLACEMENT_FRAC), int(h * PLACEMENT_FRAC)
    return [(int(rng.randint(-max_dx, max_dx + 1)), int(rng.randint(-max_dy, max_dy + 1))) for _ in range(n_trials)]


def test_scenes(png_dir, frame):
    """All calibrated PNGs except the chart photographs and the generation frame."""
    stems = sorted(p.stem for p in Path(png_dir).glob('*.png'))
    return [s for s in stems if s not in CHART_FRAMES and s != frame]


def run_eval(frame, styles, n_trials, device, png_dir, base_path, tag='_cal', bg_limit=None):
    """Score the base pattern (file base_path(style)) and the site-specific pattern of each
    style on every test scene. Writes EVAL_ROOT/<frame><tag>/results.csv."""
    out = EVAL_ROOT / (frame + tag)
    out.mkdir(parents=True, exist_ok=True)
    csvp, prog = out / 'results.csv', out / 'progress.txt'

    def log(m):
        print(m, flush=True)
        with open(prog, 'a') as f:
            f.write(m + '\n')

    t0 = time.time()
    eval_stems = test_scenes(png_dir, frame)[:bg_limit]
    camos = {s: {'site': load_rgb(CAMO_ROOT / s / (frame + tag) / 'final_camo.png'),
                 'generic': load_rgb(base_path(s))} for s in styles}
    sinet, dgnet = build_detectors(device)
    detectors = [('SINet-V2', sinet), ('DGNet', dgnet)]
    shapes = SilhouetteLibrary.CLASSES
    log(f'[start] {len(styles)} bases x {len(eval_stems)} scenes x {len(shapes)} shapes x {n_trials} positions '
        f'x 2 conditions x 2 detectors, device={device}')

    fields = ['style', 'bg', 'shape', 'trial', 'dx', 'dy', 'condition', 'detector'] + METRICS + EXTRA_COLS
    done = {(r['style'], r['bg'], r['shape']) for r in csv.DictReader(open(csvp))} if csvp.exists() else set()
    if done:
        log(f'[resume] {len(done)} cells already complete')
    else:
        with open(csvp, 'w', newline='') as f:
            csv.DictWriter(f, fieldnames=fields).writeheader()

    cell, n_cells = 0, len(styles) * len(eval_stems) * len(shapes)
    for style in styles:
        for stem in eval_stems:
            bg = load_rgb(Path(png_dir) / f'{stem}.png')
            h, w = bg.shape[:2]
            for shape in shapes:
                cell += 1
                if (style, stem, shape) in done:
                    continue
                base_mask = SilhouetteLibrary.get(shape, h, w)
                buf = []
                for t, (dx, dy) in enumerate(placements(style, stem, shape, n_trials, h, w)):
                    mask = np.roll(np.roll(base_mask, dy, 0), dx, 1)   # the same position for both conditions
                    for cond in ('generic', 'site'):
                        comp, gt = composite(camos[style][cond], bg, mask)
                        gtb = gt > 0.5
                        for dn, det in detectors:
                            pred = det.predict(comp)
                            m = CODMetrics.evaluate_all(pred, gt)
                            row = {'style': style, 'bg': stem, 'shape': shape, 'trial': t,
                                   'dx': dx, 'dy': dy, 'condition': cond, 'detector': dn}
                            row.update({k: float(m[k]) for k in METRICS})
                            row['pred_in_gt'] = float(pred[gtb].mean())
                            row['pred_out_gt'] = float(pred[~gtb].mean())
                            row['pred_max'] = float(pred.max())
                            buf.append(row)
                with open(csvp, 'a', newline='') as f:
                    csv.DictWriter(f, fieldnames=fields).writerows(buf)
                if cell % 6 == 0 or cell == n_cells:
                    log(f'[eval] cell {cell}/{n_cells} ({style}) +{(time.time()-t0)/60:.1f} min')
    log(f'[eval-done] +{(time.time()-t0)/60:.1f} min -> {csvp}')

### The color-limit test — Results, "Generation and the cost of the five-color limit"

The same gray style-transfer output of each base pattern is colored with 5, 8, 16 and 64 colors and with no limit (each pixel is mapped onto the full set of lit, non-sky colors of the photograph). The same test without style transfer uses the gray base pattern, at the same image size, colored in the same two ways. All versions are scored on the same 33 scenes, in the same positions, with the same detectors.

In [ ]:
%%module colour_limit_ablation
"""The color-limit test (Results, "Generation and the cost of the five-color limit").

The gray style-transfer output of each base pattern is held fixed and colored in five ways:

    k5            the site-specific pattern (5 colors, results/01_generation/patterns/<style>/IMG_4171_cal/final_camo.png)
    k8, k16, k64  the same k-means palette with more colors, the same recoloring by brightness rank
    kfull         no color limit: the pixel of brightness rank r takes the lit-pixel color at the
                  same quantile of the lit pixels of the generation frame, sorted by L*

The same test without style transfer uses the gray base pattern, at the size of the
style-transfer output, colored in two ways: nonst_k5 (the same five colors) and nonst_kfull.

Every arm is scored on the same 33 test scenes, 3 shapes and 20 positions as notebook 2
(the same seeds), by both detectors, so only the number of colors changes.

    build(), build_nonst()   make the patterns
    run(), run_nonst()       score them (an interrupted run continues)
    analyse()                paired tests -> results/01_generation/colour_limit/table_quant_ablation.csv
"""
import csv, gc, json, os, time
from pathlib import Path
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')
import cv2
import numpy as np

import adversarial_eval as rv
from silhouettes import SilhouetteLibrary
from cod_metrics import CODMetrics
from detectors import build_detectors
from palette import lit_only_palette, detect_sky, SHADOW_V_MAX
from recolor import recolor_frequency_weighted

ROOT = Path.cwd()          # project root (the notebook starts here)
FRAME = 'IMG_4171'
STYLES = ['multicam', 'woodland', 'realtree']
PNG_DIR = ROOT / 'images' / 'site_photos' / 'calibrated' / 'png'
CAMO = ROOT / 'results' / '01_generation' / 'patterns'
EXISTING = ROOT / 'results' / '02_adversarial_test' / 'IMG_4171_cal' / 'results.csv'   # notebook 2: the base rows
OUT = ROOT / 'results' / '01_generation' / 'colour_limit'
TABLES = OUT
KS = [8, 16, 64]
NEW_CONDS = ['k5', 'kfull', 'k16', 'k64', 'k8']
NONST_CONDS = ['nonst_kfull', 'nonst_k5']
N_TRIALS = 20
EPS = 0.03                # the margin fixed before the runs
FIELDS = ['style', 'bg', 'shape', 'trial', 'dx', 'dy', 'condition', 'detector'] + rv.METRICS + rv.EXTRA_COLS


def log(msg, prog=None):
    print(msg, flush=True)
    if prog is not None:
        with open(prog, 'a') as f:
            f.write(msg + '\n')


def eval_stems():
    return rv.test_scenes(PNG_DIR, FRAME)


def local_std(rgb, k=15):
    L = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB).astype(np.float32)[..., 0] * 100 / 255.
    L = L.astype(np.float64)
    mu, mu2 = cv2.blur(L, (k, k)), cv2.blur(L * L, (k, k))
    return float(np.sqrt(np.maximum(mu2 - mu * mu, 0)).mean())


def lit_pixels_lab(env_png):
    """Same masking as lit_only_palette: sky and deep shadow (HSV V < 60) excluded."""
    bgr = cv2.imread(str(env_png))
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2Lab).astype(np.float32)
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    lit = ~(detect_sky(rgb) | (hsv[..., 2] < SHADOW_V_MAX))
    return lab.reshape(-1, 3)[lit.ravel()]


def recolor_rank_full(gray_rgb, lit_lab):
    """Recoloring with no color limit: the pixel of brightness rank r takes the lit-pixel
    color at the same quantile of the lit pixels sorted by L*."""
    H, W, _ = gray_rgb.shape
    L = gray_rgb[..., 0].astype(np.float32).ravel()
    N = L.size
    order = np.argsort(L, kind='stable')
    ranks = np.empty(N, dtype=np.int64)
    ranks[order] = np.arange(N)
    lit_sorted = lit_lab[np.argsort(lit_lab[:, 0], kind='stable')]
    M = lit_sorted.shape[0]
    idx = np.clip(((ranks + 0.5) * M / N).astype(np.int64), 0, M - 1)
    lab_img = np.clip(lit_sorted[idx].reshape(H, W, 3), 0, 255).astype(np.uint8)
    return cv2.cvtColor(cv2.cvtColor(lab_img, cv2.COLOR_Lab2BGR), cv2.COLOR_BGR2RGB)


def pattern_path(style, cond):
    d = CAMO / style / (FRAME + '_cal')
    return d / 'final_camo.png' if cond == 'k5' else d / 'ablation' / f'{cond}.png'


def build_nonst():
    """The test without style transfer: the gray base pattern (at the size of the style-transfer
    output) colored with the stored five-color palette (nonst_k5) and with no color limit
    (nonst_kfull)."""
    env_png = PNG_DIR / f'{FRAME}.png'
    lit = lit_pixels_lab(env_png)
    meta = json.load(open(OUT / 'patterns_meta.json')) if (OUT / 'patterns_meta.json').exists() else {}
    for style in STYLES:
        d = CAMO / style / (FRAME + '_cal'); (d / 'ablation').mkdir(exist_ok=True)
        base = rv.equal_resolution_base(style)
        gray = cv2.cvtColor(cv2.cvtColor(base, cv2.COLOR_RGB2GRAY), cv2.COLOR_GRAY2RGB)
        pal = json.load(open(d / 'env_palette.json'))
        cents = np.asarray(pal['lab_centroids_sorted_by_L'], np.float32)
        fracs = np.asarray(pal['pixel_fractions_sorted_by_L'], np.float64)
        pats = {'nonst_k5': recolor_frequency_weighted(gray, cents, fracs), 'nonst_kfull': recolor_rank_full(gray, lit)}
        for cond, im in pats.items():
            cv2.imwrite(str(pattern_path(style, cond)), cv2.cvtColor(im, cv2.COLOR_RGB2BGR))
            meta[f'{style}/{cond}'] = {'source': f'grayscale base {style}.jpg {gray.shape[1]}x{gray.shape[0]}',
                                       'n_colours': int(len(np.unique(im.reshape(-1, 3), axis=0))),
                                       'texture_local_Lstd_15px': local_std(im), 'base_gray_texture': local_std(gray)}
            print(f"{style:9s} {cond:12s} colours={meta[f'{style}/{cond}']['n_colours']:7d}  texture={local_std(im):5.2f}  (base gray {local_std(gray):5.2f}, {gray.shape[1]}x{gray.shape[0]})")
    OUT.mkdir(parents=True, exist_ok=True)
    json.dump(meta, open(OUT / 'patterns_meta.json', 'w'), indent=1)


def build():
    """The k8, k16, k64 and kfull patterns from the stored gray style-transfer output."""
    env_png = PNG_DIR / f'{FRAME}.png'
    lit = lit_pixels_lab(env_png)
    meta_all = {}
    for style in STYLES:
        d = CAMO / style / (FRAME + '_cal')
        gray = np.array(__import__('PIL.Image', fromlist=['Image']).open(d / 'nst_gray.png').convert('RGB'))
        (d / 'ablation').mkdir(exist_ok=True)
        pats = {'k5': rv.load_rgb(d / 'final_camo.png')}
        for k in KS:
            cents, fracs, *_ = lit_only_palette(env_png, k=k)
            pats[f'k{k}'] = recolor_frequency_weighted(gray, cents, fracs)
            meta_all[f'{style}/k{k}'] = {'k': k, 'lab_centroids_sorted_by_L': cents.tolist(),
                                          'fractions_sorted_by_L': fracs.tolist()}
        pats['kfull'] = recolor_rank_full(gray, lit)
        meta_all[f'{style}/kfull'] = {'k': int(lit.shape[0]), 'method': 'rank-matched onto lit-pixel LAB distribution'}
        for cond, im in pats.items():
            if cond != 'k5':
                cv2.imwrite(str(pattern_path(style, cond)), cv2.cvtColor(im, cv2.COLOR_RGB2BGR))
            ncol = len(np.unique(im.reshape(-1, 3), axis=0))
            meta_all[f'{style}/{cond}'] = {**meta_all.get(f'{style}/{cond}', {}),
                                           'n_colours': ncol, 'texture_local_Lstd_15px': local_std(im),
                                           'nst_gray_texture': local_std(gray)}
            print(f'{style:9s} {cond:6s} colours={ncol:7d}  texture={local_std(im):5.2f}  (nst_gray {local_std(gray):5.2f})')
    OUT.mkdir(parents=True, exist_ok=True)
    json.dump(meta_all, open(OUT / 'patterns_meta.json', 'w'), indent=1)
    print('->', OUT / 'patterns_meta.json')


def raw_metrics(pred, gt):
    """The four standard scores (the min-max normalized scores of notebook 2 are not computed here)."""
    p = pred.astype(np.float64); g = gt.astype(np.float64)
    if p.max() > 1.01: p /= 255.0
    if g.max() > 1.01: g /= 255.0
    return {'MAE': float(CODMetrics.mae(p, g)), 'S_alpha': float(CODMetrics.s_measure(p, g)),
            'E_phi': float(CODMetrics.e_measure(p, g)), 'Fw_beta': float(CODMetrics.weighted_f_measure(p, g))}


def score_cells(conds, stems, styles, n_trials, device, csvp, prog, done=frozenset()):
    """The scoring loop of notebook 2 for the patterns of `conds`. Only the four standard
    scores are computed; the *_norm columns stay empty."""
    sinet, dgnet = build_detectors(device)
    detectors = [('SINet-V2', sinet), ('DGNet', dgnet)]
    shapes = SilhouetteLibrary.CLASSES
    pats = {(st, c): rv.load_rgb(pattern_path(st, c)) for st in styles for c in conds}
    n_cells = len(conds) * len(styles) * len(stems) * len(shapes)
    cell = 0; t0 = time.time(); started = 0
    if not csvp.exists():
        with open(csvp, 'w', newline='') as f:
            csv.DictWriter(f, fieldnames=FIELDS).writeheader()
    for cond in conds:
        for style in styles:
            for stem in stems:
                bg = rv.load_rgb(PNG_DIR / f'{stem}.png')
                h, w = bg.shape[:2]
                for shape in shapes:
                    cell += 1
                    if (cond, style, stem, shape) in done:
                        continue
                    base_mask = SilhouetteLibrary.get(shape, h, w)
                    buf = []
                    for t, (dx, dy) in enumerate(rv.placements(style, stem, shape, n_trials, h, w)):   # the positions of notebook 2
                        mask = np.roll(np.roll(base_mask, dy, 0), dx, 1)
                        comp, gt = rv.composite(pats[(style, cond)], bg, mask)
                        gtb = gt > 0.5
                        for dn, det in detectors:
                            pred = det.predict(comp)
                            m = raw_metrics(pred, gt)
                            row = {'style': style, 'bg': stem, 'shape': shape, 'trial': t, 'dx': dx, 'dy': dy,
                                   'condition': cond, 'detector': dn}
                            row.update({k: m.get(k, '') for k in rv.METRICS})
                            row['pred_in_gt'] = float(pred[gtb].mean())
                            row['pred_out_gt'] = float(pred[~gtb].mean())
                            row['pred_max'] = float(pred.max())
                            buf.append(row)
                    with open(csvp, 'a', newline='') as f:
                        csv.DictWriter(f, fieldnames=FIELDS).writerows(buf)
                    started += 1
                    # free GPU memory after each cell, so that memory does not grow in a run of hours
                    del buf; gc.collect()
                    if sinet.device.type == 'mps':
                        sinet.torch.mps.empty_cache()
                    if started % 6 == 0 or cell == n_cells:
                        el = (time.time() - t0) / 60
                        log(f'[eval] {cond} cell {cell}/{n_cells} ({style}/{stem}) +{el:.1f} min, '
                            f'ETA {el / started * (n_cells - cell):.0f} min', prog)
    log(f'[eval-done] +{(time.time()-t0)/60:.1f} min', prog)


def _done(csvp):
    return {(r['condition'], r['style'], r['bg'], r['shape']) for r in csv.DictReader(open(csvp))} if csvp.exists() else set()


def run(device):
    """Score k5, kfull, k16, k64 and k8."""
    OUT.mkdir(parents=True, exist_ok=True)
    csvp, prog = OUT / 'results.csv', OUT / 'progress.txt'
    done, stems = _done(csvp), eval_stems()
    assert len(stems) == 33, len(stems)
    log(f'[start] conds={NEW_CONDS} bgs={len(stems)} n_trials={N_TRIALS} device={device} resume={len(done)} cells done', prog)
    score_cells(NEW_CONDS, stems, STYLES, N_TRIALS, device, csvp, prog, frozenset(done))


def run_nonst(device):
    """Score nonst_kfull and nonst_k5."""
    csvp, prog = OUT / 'results.csv', OUT / 'progress.txt'
    done, stems = _done(csvp), eval_stems()
    assert len(stems) == 33, len(stems)
    log(f'[start-nonst] conds={NONST_CONDS} resume={len(done)} cells done', prog)
    score_cells(NONST_CONDS, stems, STYLES, N_TRIALS, device, csvp, prog, frozenset(done))


def analyse():
    """Paired tests over the 33 scenes (two-sided Wilcoxon, bootstrap 95 % interval) for every contrast."""
    from scipy import stats
    # only the generic-base rows come from notebook 2 (k5 is scored here), and only for
    # the backgrounds scored here; without them the contrasts against the base are skipped
    abl = list(csv.DictReader(open(OUT / 'results.csv')))
    abl_bgs = {r['bg'] for r in abl}
    rows = ([r for r in csv.DictReader(open(EXISTING)) if r['condition'] == 'generic' and r['bg'] in abl_bgs]
            if EXISTING.exists() else [])
    rows += abl
    conds = sorted({r['condition'] for r in rows})
    # placements must match across every condition within (style, bg, shape, trial)
    place = {}
    for r in rows:
        k = (r['style'], r['bg'], r['shape'], r['trial'])
        place.setdefault(k, set()).add((r['dx'], r['dy']))
    assert all(len(v) == 1 for v in place.values()), 'placement mismatch across conditions'
    bgs = sorted({r['bg'] for r in rows}); shapes = sorted({r['shape'] for r in rows})
    metrics = ['S_alpha', 'pred_in_gt', 'Fw_beta', 'E_phi', 'MAE']
    agg = {}
    for r in rows:
        for m in metrics:
            agg.setdefault((m, r['style'], r['detector'], r['shape'], r['condition'], r['bg']), []).append(float(r[m]))
    def bgvec(m, st, det, cond):
        return np.array([np.mean([np.mean(agg[(m, st, det, sh, cond, b)]) for sh in shapes]) for b in bgs])
    def level(m, st, det, cond):
        return float(np.mean([np.mean(agg[(m, st, det, sh, cond, b)]) for sh in shapes for b in bgs]))
    complete = {c for c in conds if all((('S_alpha', st, det, sh, c, b) in agg)
                for st in STYLES for det in ('SINet-V2', 'DGNet') for sh in shapes for b in bgs)}
    print('complete conditions:', sorted(complete))
    rng = np.random.default_rng(0)
    out = []
    for m in metrics:
        for st in STYLES:
            for det in ('SINet-V2', 'DGNet'):
                lv = {c: level(m, st, det, c) for c in complete}
                for c in sorted(complete):
                    out.append(dict(metric=m, style=st, detector=det, contrast=f'level:{c}', a=c, b='',
                                    delta=round(lv[c], 4), wins='', n=len(bgs), p_two_sided='', ci_lo='', ci_hi='', abs_gt_eps=''))
                pairs = [(c, 'kfull') for c in ('k5', 'k8', 'k16', 'k64') if c in complete and 'kfull' in complete]
                pairs += [('k5', c) for c in ('k8', 'k16', 'k64') if c in complete]
                pairs += [(c, 'generic') for c in ('kfull', 'k64', 'k16', 'k8') if c in complete and 'generic' in complete]
                if 'nonst_k5' in complete and 'nonst_kfull' in complete:
                    pairs += [('nonst_k5', 'nonst_kfull')]
                    if 'generic' in complete:
                        pairs += [('nonst_k5', 'generic'), ('nonst_kfull', 'generic')]
                    if 'k5' in complete and 'kfull' in complete:
                        dq_nst = bgvec(m, st, det, 'k5') - bgvec(m, st, det, 'kfull')
                        dq_non = bgvec(m, st, det, 'nonst_k5') - bgvec(m, st, det, 'nonst_kfull')
                        dd = dq_nst - dq_non
                        bs = [rng.choice(dd, len(dd), replace=True).mean() for _ in range(20000)]
                        try: p2 = float(stats.wilcoxon(dd).pvalue)
                        except ValueError: p2 = float('nan')
                        out.append(dict(metric=m, style=st, detector=det, contrast='dq_nst-dq_nonst', a='k5-kfull', b='nonst_k5-nonst_kfull',
                                        delta=round(float(dd.mean()), 4), wins=int((dd < 0).sum()), n=len(dd), p_two_sided=float(f'{p2:.3e}'),
                                        ci_lo=round(float(np.percentile(bs, 2.5)), 4), ci_hi=round(float(np.percentile(bs, 97.5)), 4), abs_gt_eps=''))
                for a, b in pairs:
                    d = bgvec(m, st, det, a) - bgvec(m, st, det, b)
                    bs = [rng.choice(d, len(d), replace=True).mean() for _ in range(20000)]
                    try:
                        p2 = float(stats.wilcoxon(d).pvalue)
                    except ValueError:
                        p2 = float('nan')
                    out.append(dict(metric=m, style=st, detector=det, contrast=f'{a}-{b}', a=a, b=b,
                                    delta=round(float(d.mean()), 4), wins=int((d < 0).sum()), n=len(d),
                                    p_two_sided=float(f'{p2:.3e}'), ci_lo=round(float(np.percentile(bs, 2.5)), 4),
                                    ci_hi=round(float(np.percentile(bs, 97.5)), 4),
                                    abs_gt_eps=bool(abs(d.mean()) > EPS) if m == 'S_alpha' else ''))
    TABLES.mkdir(parents=True, exist_ok=True)
    with open(TABLES / 'table_quant_ablation.csv', 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=list(out[0])); w.writeheader(); w.writerows(out)
    print(f'\nS_alpha, k5 - kfull (the quantization cost; >0 = quantization makes the pattern MORE detectable):')
    for r in out:
        if r['metric'] == 'S_alpha' and r['contrast'] == 'k5-kfull':
            print(f"  {r['style']:9s}{r['detector']:9s} d={r['delta']:+.3f} wins(k5 lower)={r['wins']}/33 "
                  f"p2={r['p_two_sided']:.1e} CI=[{r['ci_lo']:+.3f},{r['ci_hi']:+.3f}] |d|>eps={r['abs_gt_eps']}")
    print('\nS_alpha levels (mean over 33 bg x 3 shapes x 20 trials):')
    for st in STYLES:
        for det in ('SINet-V2', 'DGNet'):
            lv = {r['a']: r['delta'] for r in out if r['metric'] == 'S_alpha' and r['style'] == st
                  and r['detector'] == det and r['contrast'].startswith('level:')}
            print(f"  {st:9s}{det:9s} " + '  '.join(f'{c}={lv[c]:.3f}' for c in ('generic', 'k5', 'k8', 'k16', 'k64', 'kfull', 'nonst_k5', 'nonst_kfull') if c in lv))
    print('\nNo-NST control: S_alpha nonst_k5 - nonst_kfull, and (k5-kfull) - (nonst_k5-nonst_kfull) [<0 => NST reduces the quantization cost]:')
    for r in out:
        if r['metric'] == 'S_alpha' and r['contrast'] in ('nonst_k5-nonst_kfull', 'dq_nst-dq_nonst'):
            print(f"  {r['style']:9s}{r['detector']:9s} {r['contrast']:22s} d={r['delta']:+.3f} wins={r['wins']}/33 p2={r['p_two_sided']:.1e} CI=[{r['ci_lo']:+.3f},{r['ci_hi']:+.3f}]")
    print('->', TABLES / 'table_quant_ablation.csv')

### Tables for the paper

Each function reads a result file and returns one table of the paper in LaTeX. Nothing in a table is typed by hand.

In [ ]:
%%module paper_tables
"""LaTeX tables for the paper, in the format of the manuscript, made from the saved result files.

Each function reads a CSV that a notebook wrote, returns the LaTeX of one table, and
writes it to `out` when `out` is given. Nothing in a table is typed by hand.
"""
import csv
from collections import defaultdict
from pathlib import Path

import numpy as np
from scipy import stats

STYLES = ['multicam', 'woodland', 'realtree']
NAME = {'multicam': 'MultiCam', 'woodland': 'M81', 'realtree': 'Realtree'}   # 'woodland' is the M81 base
DETECTORS = ('SINet-V2', 'DGNet')


def _holm(ps):
    order = sorted(range(len(ps)), key=lambda i: ps[i])
    adj, prev = [0.0] * len(ps), 0.0
    for rank, i in enumerate(order):
        prev = max(prev, min(1.0, (len(ps) - rank) * ps[i]))
        adj[i] = prev
    return adj


def _write(tex, out):
    if out:
        out = Path(out)
        out.parent.mkdir(parents=True, exist_ok=True)
        out.write_text(tex + '\n')
    return tex


def quant_table(table_csv, out=None):
    """Table tab:quant from colour_limit/table_quant_ablation.csv (notebook 1)."""
    rows = [r for r in csv.DictReader(open(table_csv)) if r['metric'] == 'S_alpha']
    get = lambda st, det, c: next(r for r in rows if r['style'] == st and r['detector'] == det and r['contrast'] == c)
    cells = [(st, det) for st in STYLES for det in DETECTORS]
    hp = dict(zip(cells, _holm([float(get(st, det, 'dq_nst-dq_nonst')['p_two_sided']) for st, det in cells])))
    hw = dict(zip(cells, _holm([float(get(st, det, 'k5-kfull')['p_two_sided']) for st, det in cells])))
    ho = dict(zip(cells, _holm([float(get(st, det, 'nonst_k5-nonst_kfull')['p_two_sided']) for st, det in cells])))
    star = lambda v, p: f'${v:+.3f}^{{*}}$' if p < 0.05 else f'${v:+.3f}$'
    body = []
    for st in STYLES:
        for det in DETECTORS:
            w, wo, d = (get(st, det, c) for c in ('k5-kfull', 'nonst_k5-nonst_kfull', 'dq_nst-dq_nonst'))
            dv = float(d['delta'])
            # bold = reduction significant after Holm; * = dq significant after Holm (per column)
            cell = rf'$\mathbf{{{dv:+.3f}}}$' if dv < 0 and hp[(st, det)] < 0.05 else f'${dv:+.3f}$'
            body.append(f"{NAME[st]:8} & {det:8} & {star(float(w['delta']), hw[(st, det)])} & "
                        f"{star(float(wo['delta']), ho[(st, det)])} & {cell} \\\\")
    return _write(r'''\begin{table}[t]
\centering
\caption{The cost of the five-color limit. Each row is named after the base pattern; the values are about the site-specific pattern made from it. $\Delta_q$ is the $S_\alpha$ score
with five colors minus the score with no color limit; a positive value means
five colors make the pattern easier to find. Each value is the mean of the 33
scene-level differences. ``$\Delta_q$ with'' uses the
style-transfer pattern; ``$\Delta_q$ without'' repeats the test on the base pattern
alone, at the same image size. The difference is with minus without; a negative
value means style transfer reduces the cost. Two-sided Wilcoxon tests over 33
scenes, Holm correction over the six cells of each column: an asterisk marks a
significant $\Delta_q$, bold a significant reduction.}
\label{tab:quant}
\small
\setlength{\tabcolsep}{4pt}
\begin{tabular}{llrrr}
\hline
Base & Detector & $\Delta_q$ with & $\Delta_q$ without & difference \\
\hline
''' + '\n'.join(body) + r'''
\hline
\end{tabular}
\end{table}''', out)


def base_table(results_csv, out=None):
    """Table tab:base from 02_adversarial_test/IMG_4171_cal/results.csv (notebook 2)."""
    agg = defaultdict(list)
    for r in csv.DictReader(open(results_csv)):
        agg[(r['bg'], r['style'], r['detector'], r['shape'], r['condition'])].append(float(r['S_alpha']))
    bgs = sorted({k[0] for k in agg}); shapes = sorted({k[3] for k in agg})
    cells = [(st, det) for st in STYLES for det in DETECTORS]
    D = {(st, det): np.array([np.mean([np.mean(agg[(b, st, det, sh, 'site')]) - np.mean(agg[(b, st, det, sh, 'generic')])
                                       for sh in shapes]) for b in bgs]) for st, det in cells}
    H = dict(zip(cells, _holm([stats.wilcoxon(D[c]).pvalue for c in cells])))
    holm_note = ("``better'' and ``worse'' are significant after Holm correction over the six tests; "
                 "``no difference'' is not")
    body = []
    for st, det in cells:
        d = D[(st, det)]
        res = ('no difference' if H[(st, det)] >= 0.05 else
               'better' if d.mean() < 0 else r'\textbf{worse}')
        hp_ = H[(st, det)]
        ptxt = '$<0.001$' if hp_ < 0.001 else f'${hp_:.2f}$'
        body.append(f"{NAME[st]:8} & {det:8} & ${d.mean():+.3f}$ & {int((d < 0).sum())}/{len(d)} & {ptxt} & {res} \\\\")
    return _write(r'''\begin{table}[t]
\centering
\caption{Each site-specific pattern against the base pattern it was made from.
Each row is named after the base pattern. $\Delta S_\alpha$ is the mean of the 33 scene-level differences, site-specific
minus base, so a negative value means the site-specific pattern is harder to find. The base is downsampled to the size of the
site-specific pattern. ``Wins'' counts how many of the 33
scenes favor it. Tests are two-sided Wilcoxon signed-rank; ''' + holm_note + r'''.}
\label{tab:base}
\footnotesize
\setlength{\tabcolsep}{3pt}
\begin{tabular}{llrrrl}
\hline
Base & Detector & $\Delta S_\alpha$ & Wins & Holm $p$ & Result \\
\hline
''' + '\n'.join(body) + r'''
\hline
\end{tabular}
\end{table}''', out)


FIELD_LABEL = {'print_v1': r'\texttt{print\_v1} (pipeline)', 'print_v2': r'\texttt{print\_v2}',
               'multicam_ocp': 'MultiCam', 'woodland_m81': 'M81, view 1', 'woodland_m81_b': 'M81, view 2',
               'hunting_realtree': 'Realtree'}
FIELD_ORDER = ['print_v1', 'print_v2', 'multicam_ocp', 'woodland_m81', 'woodland_m81_b', 'hunting_realtree']


def location_table(results_csv, site, out=None):
    """Table tab:locA (site1) or tab:locB (site2) from 03_field_test/two_sites/results.csv (notebook 3)."""
    rows = [r for r in csv.DictReader(open(results_csv)) if r['site'] == site]
    fmt = lambda v: '$<0.01$' if v < 0.01 else f'${v:.2f}$'
    body = []
    for label in [l for l in FIELD_ORDER if any(r['label'] == l for r in rows)]:
        vals = []
        for kind in ('full', 'best'):
            for det in DETECTORS:
                r = [float(x['pred_in_gt']) for x in rows if x['label'] == label and x['detector'] == det
                     and (kind == 'best' or x['condition'] == 'full')]
                vals.append(fmt(max(r)))
        name = FIELD_LABEL[label] if site == 'site2' else FIELD_LABEL[label].replace(', view 1', '')
        body.append(f'{name} & ' + ' & '.join(vals) + r' \\')
    n = 'AB'[int(site[-1]) - 1]
    if site == 'site1':
        cap = (r'Location~A. Mean detector score inside the traced outline; lower means harder to find. '
               r"``Max'' is the highest detector score over the full frame and crops of $1.5\times$, $2.5\times$ and "
               r'$4\times$ the item size. \texttt{print\_v2} is not pipeline output.')
    else:
        cap = (r'Location~B, same measures as Table~\ref{tab:locA}. The two M81 rows are two photographs '
               r'of one unmoved drape, taken 17\,s apart.')
    return _write(r'''\begin{table}[t]
\centering
\caption{''' + cap + r'''}
\label{tab:loc''' + n + r'''}
\footnotesize
\setlength{\tabcolsep}{4pt}
\begin{tabular}{lrrrr}
\hline
 & \multicolumn{2}{c}{Full frame} & \multicolumn{2}{c}{Max} \\
Item & SINet-V2 & DGNet & SINet-V2 & DGNet \\
\hline
''' + '\n'.join(body) + r'''
\hline
\end{tabular}
\end{table}''', out)


def recolor_table(site_csv, ablation_csv, out=None):
    """Table tab:recolor: how much color alone does (notebook 2, uses the notebook 1 ablation rows).

    recolored = base pattern recolored with the five site colors, no style transfer (nonst_k5);
    all arms share scenes, shapes and positions, so each column is a paired difference."""
    agg = defaultdict(list)
    for f in (site_csv, ablation_csv):
        for r in csv.DictReader(open(f)):
            if r['condition'] in ('site', 'generic', 'nonst_k5', 'kfull'):
                agg[(r['bg'], r['style'], r['detector'], r['shape'], r['condition'])].append(float(r['S_alpha']))
    bgs = sorted({k[0] for k in agg}); shapes = sorted({k[3] for k in agg})
    vec = lambda st, det, c: np.array([np.mean([np.mean(agg[(b, st, det, sh, c)]) for sh in shapes]) for b in bgs])
    cells = [(st, det) for st in STYLES for det in DETECTORS]
    cols = [('nonst_k5', 'generic'), ('site', 'nonst_k5'), ('kfull', 'generic')]
    D = {(c, k): vec(*c, k[0]) - vec(*c, k[1]) for c in cells for k in cols}
    H = {k: dict(zip(cells, _holm([stats.wilcoxon(D[(c, k)]).pvalue for c in cells]))) for k in cols}
    body = []
    for c in cells:
        vals = [f'${D[(c, k)].mean():+.3f}^{{*}}$' if H[k][c] < 0.05 else f'${D[(c, k)].mean():+.3f}$' for k in cols]
        body.append(f"{NAME[c[0]]:8} & {c[1]:8} & " + ' & '.join(vals) + r' \\')
    return _write(r'''\begin{table}[t]
\centering
\caption{What color alone does. Each row is named after the base pattern. Each column is a paired $\Delta S_\alpha$ on the same
33 scenes and positions, given as the mean of the 33 scene-level differences; a negative value means the first pattern is harder to
find. ``Recolored'' is the base pattern recolored with the five site colors, with
no style transfer; ``site'' is the site-specific pattern; ``no limit'' is the
style-transfer pattern colored with no color limit. An asterisk marks a
difference that is significant after Holm correction over the six cells of its
column (two-sided Wilcoxon).}
\label{tab:recolor}
\footnotesize
\setlength{\tabcolsep}{3pt}
\begin{tabular}{llrrr}
\hline
 & & recolored & site & no limit \\
Base & Detector & $-$ base & $-$ recolored & $-$ base \\
\hline
''' + '\n'.join(body) + r'''
\hline
\end{tabular}
\end{table}''', out)

## Step 0 — color calibration

The chart photographs get no PNG, because they are never test scenes.

In [ ]:
import calibration
calibration.calibrate(str(SITE.relative_to(ROOT)), CHART, str(PNG_DIR.relative_to(ROOT)), exclude=CHART_FRAMES)

## Step 1 — the luminance map of the generation frame (figure `fig_luminance`)

(a) The generation frame, (b) its nine tiles in gray, (c) their average after histogram equalization, which is the content image for style transfer.

In [ ]:
from luminance import segment_image, luminance_map
frame_png = PNG_DIR/f'{FRAME}.png'
env = Image.open(frame_png).convert('RGB')
tiles = [t[0].convert('L') for t in segment_image(env, 3, 3)]
lum = luminance_map(env, rows=3, cols=3)
fig = plt.figure(figsize=(7.2, 3.6))
gs = fig.add_gridspec(3, 9, wspace=0.08, hspace=0.08)
ax = fig.add_subplot(gs[:, 0:3]); ax.imshow(env); ax.axis('off'); ax.set_title('(a) generation frame', fontsize=9)
for i, t in enumerate(tiles):
    a = fig.add_subplot(gs[i // 3, 3 + i % 3]); a.imshow(t, cmap='gray'); a.axis('off')
    if i == 1: a.set_title('(b) nine gray tiles', fontsize=9)
ax = fig.add_subplot(gs[:, 6:9]); ax.imshow(lum, cmap='gray'); ax.axis('off'); ax.set_title('(c) luminance map', fontsize=9)
plt.savefig(FIG/'fig_luminance.png', dpi=200, bbox_inches='tight'); plt.show()
print('->', FIG/'fig_luminance.png')

## Step 2 — the three site-specific patterns (figure `fig_patterns`)

Top row of the figure: the three base patterns. Bottom row: the site-specific pattern made from each, all in the same five colors from the lit, non-sky pixels of `IMG_4171`.

In [ ]:
from generate_pattern import generate
for style in STYLES:
    print(f'--- {NAME[style]} ({style})')
    generate(frame_png, style, ITERS, PATTERNS/style/(FRAME+TAG))

import matplotlib as mpl
PAPER_STYLE = {'font.family': 'serif', 'font.serif': ['STIXGeneral', 'Times New Roman', 'DejaVu Serif'],
               'mathtext.fontset': 'stix', 'font.size': 8.0, 'text.color': '#1A1A1A', 'axes.labelcolor': '#1A1A1A',
               'figure.facecolor': 'white', 'axes.facecolor': 'white', 'savefig.facecolor': 'white'}
with mpl.rc_context(PAPER_STYLE):
    fig = plt.figure(figsize=(3.4967, 2.45), layout='constrained')      # the width of one paper column
    fig.get_layout_engine().set(w_pad=0.012, h_pad=0.012, wspace=0.01, hspace=0.02)
    gs = fig.add_gridspec(2, 3, hspace=0.04, wspace=0.05)
    for col, st in enumerate(('woodland', 'multicam', 'realtree')):
        base = cv2.imread(str(ROOT/'images'/'base_patterns'/f'{st}.jpg'))
        site = cv2.imread(str(PATTERNS/st/(FRAME+TAG)/'final_camo.png'))
        for r, im in enumerate([base, site]):
            h, w = im.shape[:2]; side = min(h, w)                        # center square, 400 x 400
            im = cv2.resize(im[(h - side)//2:(h + side)//2, (w - side)//2:(w + side)//2], (400, 400), interpolation=cv2.INTER_AREA)
            ax = fig.add_subplot(gs[r, col]); ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
            ax.set_xticks([]); ax.set_yticks([])
            for sp in ax.spines.values(): sp.set_visible(False)
            if r == 0: ax.set_title({'woodland': 'M81 woodland', 'multicam': 'MultiCam', 'realtree': 'Realtree'}[st], fontsize=8, pad=2)
            if col == 0: ax.set_ylabel('off-the-shelf base' if r == 0 else 'site-specific', fontsize=8, labelpad=3)
    fig.savefig(FIG/'fig_patterns.png', dpi=400, pil_kwargs={'optimize': True})
    plt.show()
print('->', FIG/'fig_patterns.png')

## Step 3 — local contrast at each stage

Local contrast = mean standard deviation of L\* in a 15 px window. The recoloring step assigns colors by brightness rank, so the pattern cannot carry more contrast than the palette spans. The table also gives the L\* of the five palette colors.

The background value is measured on the held-out site photographs, on the whole frame, with the same statistic.

In [ ]:
def lum_L(rgb):
    return cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB).astype(np.float32)[..., 0] * 100/255.

def local_contrast(rgb, k=15):
    L = lum_L(rgb).astype(np.float64)
    mu, mu2 = cv2.blur(L, (k, k)), cv2.blur(L*L, (k, k))
    return float(np.sqrt(np.maximum(mu2 - mu*mu, 0)).mean())

def rgb(p):
    return np.asarray(Image.open(p).convert('RGB'))

held_out = [p for p in sorted(PNG_DIR.glob('*.png')) if p.stem not in CHART_FRAMES and p.stem != FRAME]
bg = [local_contrast(rgb(p)) for p in held_out]
print(f'held-out backgrounds (n={len(bg)}): local contrast {min(bg):.1f} to {max(bg):.1f}, median {np.median(bg):.1f}\n')

print(f"{'base':10}{'luminance map':>15}{'style transfer':>16}{'five colours':>14}   palette L* (sorted)")
for style in STYLES:
    d = PATTERNS/style/(FRAME+TAG)
    Ls = np.sort(np.array(json.load(open(d/'env_palette.json'))['lab_centroids_sorted_by_L'])[:, 0] * 100/255.)
    t = [local_contrast(rgb(d/fn)) for fn in ('luminance_map.png', 'nst_gray.png', 'final_camo.png')]
    print(f"{NAME[style]:10}{t[0]:15.1f}{t[1]:16.1f}{t[2]:14.1f}   {np.round(Ls, 1)}")

## Step 4 — the color-limit test (table `tab:quant`)

`build()` and `build_nonst()` make the patterns, `run()` and `run_nonst()` score them (an interrupted run continues when `FRESH_RUN = False`), and `analyse()` makes the paired tests.

In [ ]:
import colour_limit_ablation as cla
if FRESH_RUN and cla.OUT.exists():
    shutil.rmtree(cla.OUT)
cla.build()            # k8, k16, k64 and no limit, from each style-transfer output
cla.build_nonst()      # 5 colors and no limit, from the base pattern without style transfer
cla.run(DEVICE)        # scores k5, kfull, k16, k64, k8
cla.run_nonst(DEVICE)  # scores nonst_k5, nonst_kfull
cla.analyse()          # -> results/01_generation/colour_limit/table_quant_ablation.csv

### Table — the cost of the five-color limit

$\Delta_q$ = $S_\alpha$ with five colors − $S_\alpha$ with no color limit (positive: five colors make the pattern easier to find). "difference" = $\Delta_q$ with style transfer − $\Delta_q$ without it (negative: style transfer reduces the cost). Two-sided Wilcoxon signed-rank tests over 33 backgrounds; Holm correction over each column of six tests.

In [ ]:
T = [r for r in csv.DictReader(open(cla.OUT/'table_quant_ablation.csv')) if r['metric'] == 'S_alpha']
def row(st, det, contrast):
    return next(r for r in T if r['style'] == st and r['detector'] == det and r['contrast'] == contrast)

def holm(ps):
    order = sorted(range(len(ps)), key=lambda i: ps[i]); adj = [0.0]*len(ps); prev = 0.0
    for rank, i in enumerate(order):
        prev = max(prev, min(1.0, (len(ps) - rank) * ps[i])); adj[i] = prev
    return adj

cells = [(st, det) for st in STYLES for det in DETECTORS]
q  = {c: row(*c, 'k5-kfull') for c in cells}
nq = {c: row(*c, 'nonst_k5-nonst_kfull') for c in cells}
dd = {c: row(*c, 'dq_nst-dq_nonst') for c in cells}
hq  = dict(zip(cells, holm([float(q[c]['p_two_sided']) for c in cells])))
hdd = dict(zip(cells, holm([float(dd[c]['p_two_sided']) for c in cells])))

print(f"{'base':9}{'detector':10}{'dq with':>9}{'p':>9}{'Holm':>8}{'dq without':>12}{'difference':>12}{'p':>9}{'Holm':>8}")
for c in cells:
    print(f"{NAME[c[0]]:9}{c[1]:10}{float(q[c]['delta']):+9.3f}{float(q[c]['p_two_sided']):9.1e}{hq[c]:8.1e}"
          f"{float(nq[c]['delta']):+12.3f}{float(dd[c]['delta']):+12.3f}{float(dd[c]['p_two_sided']):9.1e}{hdd[c]:8.1e}")

lv = lambda st, det, k: float(row(st, det, f'level:{k}')['delta'])
print('\nshare of the five-colour cost paid between 5 and 8 colours, and cost left at 64 colours:')
for c in cells:
    cost = lv(*c, 'k5') - lv(*c, 'kfull')
    share = (lv(*c, 'k5') - lv(*c, 'k8')) / cost if abs(cost) > 1e-9 else float('nan')
    print(f"  {NAME[c[0]]:9}{c[1]:10} cost {cost:+.3f}   5->8 share {share:6.0%}   k64 - kfull {lv(*c, 'k64') - lv(*c, 'kfull'):+.3f}")

### The same test with the base pattern at its original size

The paper states that with the base pattern at its original 1600×1600 size, the test without style transfer gave support in only three cells. That run (2026-09-24, commit `c609b81`) is kept in `results/native_resolution_2026-09-24/`.

In [ ]:
NAT = ROOT/'results'/'native_resolution_2026-09-24'/'table_quant_ablation.csv'
nat = [r for r in csv.DictReader(open(NAT)) if r['metric'] == 'S_alpha']
nrow = lambda st, det, c: next(r for r in nat if r['style'] == st and r['detector'] == det and r['contrast'] == c)
print(f"{'base':9}{'detector':10}{'without: equal':>16}{'native':>9}{'difference: equal':>19}{'native':>9}")
for c in cells:
    de, dn = float(dd[c]['delta']), float(nrow(*c, 'dq_nst-dq_nonst')['delta'])
    print(f"{NAME[c[0]]:9}{c[1]:10}{float(nq[c]['delta']):+16.3f}{float(nrow(*c, 'nonst_k5-nonst_kfull')['delta']):+9.3f}"
          f"{de:+19.3f}{dn:+9.3f}{'   sign changed' if de * dn < 0 else ''}")

## Table `tab:quant` for the paper

The cell writes the LaTeX of the table to `results/01_generation/tables/`.

In [ ]:
import paper_tables as pt
TAB = ROOT/'results'/'01_generation'/'tables'
print(pt.quant_table(cla.OUT/'table_quant_ablation.csv', out=TAB/'tab_quant.tex'))

## Answer — research question and hypothesis half (a)

This cell applies the rules from the top of the notebook to the table above. It prints a verdict for each cell and a summary.

**Resolution.** Both arms have the same image size: the base was cropped to the shape 3:4 (75 % of its width) and reduced to the 400 × 300 size of the style-transfer output. A run with the base at its native 1600 × 1600 size gave half (a) in 3 of 6 cells; the comparison cell above shows the difference.

In [ ]:
ALPHA = 0.05
print('RESEARCH QUESTION - how much does the five-colour limit cost?')
cost = [c for c in cells if float(q[c]['delta']) > 0 and hq[c] < ALPHA]
for c in cells:
    d = float(q[c]['delta'])
    print(f"  {NAME[c[0]]:9}{c[1]:10} dq = {d:+.3f}  Holm p = {hq[c]:.1e}  -> "
          + ('real cost' if c in cost else 'no detectable cost'))
if cost:
    v = [float(q[c]['delta']) for c in cost]
    print(f'ANSWER: the limit has a real cost in {len(cost)} of {len(cells)} cells, from {min(v):+.3f} to {max(v):+.3f} in S_alpha;'
          f" no detectable cost in {', '.join(NAME[c[0]] + '/' + c[1] for c in cells if c not in cost)}.")
else:
    print('ANSWER: no cell shows a real cost of the five-colour limit.')

print('\nHYPOTHESIS HALF (a) - style transfer compensates for most of the loss')
supported = []
for c in cells:
    w, wo, diff = float(q[c]['delta']), float(nq[c]['delta']), float(dd[c]['delta'])
    share = 1 - w / wo if wo > 0 else float('nan')
    reduces = diff < 0 and hdd[c] < ALPHA
    ok = reduces and share > 0.5
    supported += [c] if ok else []
    why = ('supported' if ok else
           'not supported: the reduction is not significant after Holm' if not reduces else
           'not supported: style transfer removes less than half of the cost')
    print(f"  {NAME[c[0]]:9}{c[1]:10} dq with {w:+.3f}  without {wo:+.3f}  share compensated {share:5.0%}  "
          f"difference {diff:+.3f} (Holm p {hdd[c]:.1e})  -> {why}")
print(f'ANSWER: half (a) is supported in {len(supported)} of {len(cells)} cells'
      + (f" ({', '.join(NAME[c[0]] + '/' + c[1] for c in supported)})" if supported else '') + '.')

## Appendix — code that made two inputs earlier

The base patterns in `images/base_patterns/` and the printed sheet `print_v1` are inputs of the study. The cells below hold the code that made them.

### Base patterns — Methodology, "Synthesis: Style Transfer and Recoloring"

The three base patterns (MultiCam, M81 and Realtree) are photographs of the commercial garments, cropped square, resized to 1600 px and corrected for uneven light.

This notebook defines the code but does not start it. `prepare_base_patterns.make_base_patterns()` makes the three files again from the RAW photographs in `images/base_patterns/raw/`; it writes over the files in `images/base_patterns/`, and all results then change. `make_base_patterns(write=False)` only prints the numbers.

In [ ]:
%%module prepare_base_patterns
"""The three base patterns, made from RAW photographs of the commercial garments.

    IMG_4264.DNG  MultiCam ripstop        -> multicam.jpg
    IMG_4266.DNG  Realtree hunting print  -> realtree.jpg
    IMG_4268.DNG  M81 woodland            -> woodland.jpg

Each photograph is decoded with rawpy (camera white balance, sRGB), cropped to a square that
leaves out the brand mark, bright reflections, collars and edges, resized to 1600 px, and
corrected for uneven light: L* is divided by a strongly blurred copy of itself. The pipeline
uses the base pattern in gray, so a fold shadow would otherwise become part of the style.
"""
from pathlib import Path

import cv2
import numpy as np
import rawpy

ROOT = Path.cwd()          # project root (the notebook starts here)
SRC_DIR = ROOT / 'images' / 'base_patterns' / 'raw'
OUT_DIR = ROOT / 'images' / 'base_patterns'

SIDE = 1600
SIGMA_FRAC = 0.25          # blur of the light correction, as a fraction of SIDE

# style -> (source stem, crop x0, y0, size) in developed-frame pixels
SOURCES = {
    'multicam': ('IMG_4264', 600, 600, 2200),
    'realtree': ('IMG_4266', 200, 1150, 2600),
    'woodland': ('IMG_4268', 150, 350, 2600),
}


def develop(stem: str) -> np.ndarray:
    with rawpy.imread(str(SRC_DIR / f'{stem}.DNG')) as raw:
        rgb = raw.postprocess(use_camera_wb=True, no_auto_bright=False,
                              output_bps=8, output_color=rawpy.ColorSpace.sRGB)
    return cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)


def flatten_illumination(bgr: np.ndarray) -> np.ndarray:
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB).astype(np.float32)
    L = lab[:, :, 0]
    illum = cv2.GaussianBlur(L, (0, 0), SIGMA_FRAC * bgr.shape[0])
    lab[:, :, 0] = np.clip(L * illum.mean() / np.maximum(illum, 1e-3), 0, 255)
    return cv2.cvtColor(lab.astype(np.uint8), cv2.COLOR_LAB2BGR)


def illum_spread(bgr: np.ndarray) -> float:
    """Spread of the blurred brightness, in % of the mean: how even the light is."""
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY).astype(np.float32)
    illum = cv2.GaussianBlur(gray, (0, 0), 400)
    return 100 * (np.percentile(illum, 95) - np.percentile(illum, 5)) / illum.mean()


def build(style: str) -> np.ndarray:
    stem, x0, y0, size = SOURCES[style]
    frame = develop(stem)
    h, w = frame.shape[:2]
    if y0 + size > h or x0 + size > w:
        raise ValueError(f'{style}: crop ({x0},{y0},{size}) exceeds {stem} at {w}x{h}')
    crop = frame[y0:y0 + size, x0:x0 + size]
    crop = cv2.resize(crop, (SIDE, SIDE), interpolation=cv2.INTER_AREA)
    return flatten_illumination(crop)


def make_base_patterns(write=True):
    """Make the three base patterns. write=True writes over the files in images/base_patterns/."""
    for style in SOURCES:
        out = build(style)
        print(f'{style:9s} <- {SOURCES[style][0]}.DNG  {out.shape[1]}x{out.shape[0]}  '
              f'illumination spread {illum_spread(out):.1f}%')
        if write:
            cv2.imwrite(str(OUT_DIR / f'{style}.jpg'), out, [cv2.IMWRITE_JPEG_QUALITY, 95])

### `print_v1` — Methodology, "Fabrication"

The small tile was not printed side by side many times, because the joins would show and the pattern would visibly repeat. Instead, style transfer was run once more at the size of the whole sheet, which produced one continuous image with no repeats. That image is not identical to the small tile.

This notebook defines the code but does not start it. `generate_sheet.generate_sheet('woodland')` makes the sheet again from the palette of Step 2 and writes it to `results/01_generation/sheets/` and `results/01_generation/print_files/`. The file that went to the printer was not kept; `images/printed_sheets/` holds the file of 2026-09-02.

In [ ]:
%%module generate_sheet
"""print_v1: the site-specific M81 pattern made again at the size of the whole printed sheet.

The small 400 x 300 tile is not printed side by side, because the joins would show and the
pattern would visibly repeat. Instead, style transfer runs once more on a 1376 x 1408 canvas,
which gives one continuous image with no repeats. The content image is the luminance map of
the generation frame (a center crop, enlarged to the canvas). The style targets are the Gram
matrices of the gray base pattern at the tile size, so the texture scale of the sheet matches
the tile. The weights are scaled to the canvas size. The gray result is recolored with the
five colors of the tile (env_palette.json of notebook 1), enlarged 4x, and every pixel is set
to the nearest of the five colors: the print file is 28.16 x 27.52 in at 200 dpi.

The sheet is not identical to the tile, so the field test evaluates the design on cloth, not
the exact tile that the detectors scored.
"""
import json
from pathlib import Path

import cv2
import numpy as np
from PIL import Image

from luminance import luminance_map
from recolor import recolor_frequency_weighted
from generate_pattern import build_feature_extractor, grayscale_camo

ROOT = Path.cwd()          # project root (the notebook starts here)
CANVAS_H, CANVAS_W = 1376, 1408
STYLE_W, STYLE_H = 300, 400
SEED = 42
CONTENT_LAYER = 'block5_conv2'
STYLE_LAYERS = ['block1_conv1', 'block2_conv1', 'block3_conv1', 'block4_conv1', 'block5_conv1']
CANONICAL_CONTENT_WEIGHT = 2.5e-8
STYLE_WEIGHT = 1e-6
# for plain gradient descent, a larger learning rate has the same effect as a larger loss
LOSS_SCALE = (CANVAS_H * CANVAS_W) / (STYLE_H * STYLE_W)
CONTENT_PNG = ROOT / 'images' / 'site_photos' / 'calibrated' / 'png' / 'IMG_4171.png'
BASE_DIR = ROOT / 'images' / 'base_patterns'
CAMO_ROOT = ROOT / 'results' / '01_generation' / 'patterns'
SHEET_ROOT = ROOT / 'results' / '01_generation' / 'sheets'
PRINT_ROOT = ROOT / 'results' / '01_generation' / 'print_files'


def force_grayscale(rgb):
    return cv2.cvtColor(cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY), cv2.COLOR_GRAY2RGB)


def _preprocess_array(tf, vgg19, rgb):
    return tf.convert_to_tensor(vgg19.preprocess_input(np.asarray(rgb, dtype=np.float32)[None]))


def loss_config(tf, vgg19, extractor):
    """Content weight and per-layer style weights, scaled from the tile to the canvas."""
    def areas(h, w):
        feats = extractor(_preprocess_array(tf, vgg19, np.zeros((h, w, 3), dtype=np.uint8)))
        return {n: int(feats[n].shape[1]) * int(feats[n].shape[2]) for n in [CONTENT_LAYER] + STYLE_LAYERS}
    tile, canvas = areas(STYLE_H, STYLE_W), areas(CANVAS_H, CANVAS_W)
    size = CANVAS_H * CANVAS_W
    return {'content_weight': CANONICAL_CONTENT_WEIGHT * tile[CONTENT_LAYER] / canvas[CONTENT_LAYER],
            'per_layer': {n: {'n_l': tile[n], 'N_l': canvas[n],
                              'w_l': (STYLE_WEIGHT / len(STYLE_LAYERS)) * (canvas[n] / size) ** 2 / 36.0}
                          for n in STYLE_LAYERS}}


def content_image():
    """Center crop (504 x 493) of the luminance map of the generation frame, enlarged to the canvas."""
    lum = luminance_map(Image.open(CONTENT_PNG).convert('RGB'), rows=3, cols=3)
    left, top = (lum.width - 504) // 2, (lum.height - 493) // 2
    return lum.crop((left, top, left + 504, top + 493)).resize((CANVAS_W, CANVAS_H)).convert('RGB')


def style_targets(tf, vgg19, extractor, standalone, content, cfg):
    """Gram matrix of each style layer of the gray base pattern at the tile size, divided by the
    number of feature positions."""
    test_content = np.asarray(Image.fromarray(content).resize((STYLE_W, STYLE_H)), dtype=np.uint8)
    style_tensor = _preprocess_array(tf, vgg19, standalone)
    combination_tensor = _preprocess_array(tf, vgg19, test_content)
    features = extractor(tf.concat([combination_tensor, style_tensor, combination_tensor], axis=0))
    targets = {}
    for name in STYLE_LAYERS:
        style_feature = features[name].numpy().astype(np.float64, copy=False)[1]
        style_flat = style_feature.reshape(-1, style_feature.shape[-1]).T
        n_l = style_flat.shape[1]
        assert n_l == cfg['per_layer'][name]['n_l'], name
        targets[name] = tf.constant((style_flat @ style_flat.T / np.float64(n_l)).astype(np.float32), dtype=tf.float32)
    return targets


def run_nst(tf, vgg19, extractor, content, targets, cfg, iters):
    from keras.optimizers import SGD

    def gram(x):
        x = tf.transpose(x, (2, 0, 1))
        flat = tf.reshape(x, (tf.shape(x)[0], -1))
        return tf.matmul(flat, tf.transpose(flat))

    def terms(combination, base):
        feats = extractor(tf.concat([base, combination], axis=0))
        cf = feats[CONTENT_LAYER]
        content_term = cfg['content_weight'] * tf.reduce_sum(tf.square(cf[1] - cf[0]))
        style_term = tf.constant(0.0)
        for name in STYLE_LAYERS:
            N_l, w_l = cfg['per_layer'][name]['N_l'], cfg['per_layer'][name]['w_l']
            normalized_gram = gram(feats[name][1]) / tf.cast(N_l, tf.float32)
            style_term += w_l * tf.reduce_sum(tf.square(targets[name] - normalized_gram))
        return content_term, style_term

    @tf.function   # keep tf.function: the eager path gave wrong results on the Apple GPU
    def loss_and_grads(combination, base):
        with tf.GradientTape() as tape:
            content_term, style_term = terms(combination, base)
            loss = content_term + style_term
        return loss, content_term, style_term, tape.gradient(loss, combination)

    base = _preprocess_array(tf, vgg19, content)
    combination = tf.Variable(base)
    optimizer = SGD(tf.keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=100.0 * LOSS_SCALE, decay_steps=100, decay_rate=0.96))
    for i in range(1, iters + 1):
        loss, content_term, style_term, grads = loss_and_grads(combination, base)
        optimizer.apply_gradients([(grads, combination)])
        if i == 1 or i % 100 == 0:
            print(f'  iter {i:5d}/{iters}: loss={float(loss.numpy()):.6g}', flush=True)
    value = combination.numpy().reshape((CANVAS_H, CANVAS_W, 3)).copy()
    value[:, :, 0] += 103.939; value[:, :, 1] += 116.779; value[:, :, 2] += 123.68
    return np.clip(value[:, :, ::-1], 0, 255).astype(np.uint8)


def snap_to_palette(rgb, palette_rgb):
    """Set every pixel to the nearest palette color (RGB distance)."""
    flat = rgb.reshape(-1, 3).astype(np.float32)
    palette = palette_rgb.astype(np.float32)
    labels = np.empty(len(flat), dtype=np.uint8)
    for start in range(0, len(flat), 250_000):
        block = flat[start:start + 250_000]
        labels[start:start + 250_000] = np.argmin(((block[:, None, :] - palette[None, :, :]) ** 2).sum(axis=2), axis=1)
    return palette_rgb[labels].reshape(rgb.shape)


def generate_sheet(style='woodland', iters=2000):
    """Make the sheet for one base pattern. Returns the path of the print file."""
    out = SHEET_ROOT / style / 'IMG_4171_cal'
    out.mkdir(parents=True, exist_ok=True)
    tf, vgg19, extractor = build_feature_extractor()
    np.random.seed(SEED); tf.random.set_seed(SEED)
    cfg = loss_config(tf, vgg19, extractor)

    content_pil = content_image()
    content_pil.save(out / 'luminance_map.png')
    standalone = np.asarray(Image.fromarray(grayscale_camo(BASE_DIR / f'{style}.jpg')).resize((STYLE_W, STYLE_H)), dtype=np.uint8)
    content = np.asarray(content_pil, dtype=np.uint8)
    targets = style_targets(tf, vgg19, extractor, standalone, content, cfg)
    nst_gray = force_grayscale(run_nst(tf, vgg19, extractor, content, targets, cfg, iters))
    Image.fromarray(nst_gray).save(out / 'nst_gray.png')

    pal = json.load(open(CAMO_ROOT / style / 'IMG_4171_cal' / 'env_palette.json'))
    cents = np.asarray(pal['lab_centroids_sorted_by_L'], dtype=np.float32)
    fracs = np.asarray(pal['pixel_fractions_sorted_by_L'], dtype=np.float64)
    final = recolor_frequency_weighted(nst_gray, cents, fracs)
    Image.fromarray(final).save(out / 'final_sheet.png')

    palette_rgb = np.array([cv2.cvtColor(np.uint8([[np.clip(c, 0, 255)]]), cv2.COLOR_Lab2RGB)[0, 0] for c in cents], dtype=np.uint8)
    enlarged = cv2.resize(final, (CANVAS_W * 4, CANVAS_H * 4), interpolation=cv2.INTER_LANCZOS4)
    PRINT_ROOT.mkdir(parents=True, exist_ok=True)
    print_path = PRINT_ROOT / f'victoria_{style}_IMG_4171_cal_fullbleed_unique_28.16x27.52in_200dpi.png'
    Image.fromarray(snap_to_palette(enlarged, palette_rgb), mode='RGB').save(print_path, dpi=(200, 200))
    print(f'-> {print_path}')
    return print_path